Copia histórica sanitizada. No ejecutar para el flujo nuevo: usa EmotiveAD_Colab.ipynb.
Se eliminaron credenciales y salidas; el código histórico conserva sus incompatibilidades.

In [ ]:
import os

# EmotivaAD
Este notebook sube videos a Google Drive, extrae audio, realiza diarización con pyannote.ai y genera análisis de segmentos silenciosos con detección de escenas.

In [ ]:
# @title
# Instalar dependencias necesarias
!apt-get -y update && apt-get -y install ffmpeg
!pip -q install moviepy pydub pyannoteai-sdk==0.4.0 scenedetect==0.6.4 opencv-python==4.9.0.80
!pip -q install google-api-python-client google-auth-httplib2 google-auth-oauthlib
!pip install pyannoteai-sdk
!pip install scenedetect>=0.6.2

print("✅ Dependencias instaladas correctamente")

In [ ]:
import numpy;
print(numpy.__version__)

In [ ]:
# @title
# Downgrade NumPy to a compatible version for libraries like OpenCV
!pip install numpy==1.23.5 --force-reinstall

print('✅ NumPy downgraded to 1.23.5')

In [ ]:
# @title
import os
import time
import subprocess
import json
import shutil
import getpass
from pathlib import Path
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from google.colab import files
from moviepy.video.io.VideoFileClip import VideoFileClip
from pydub import AudioSegment
from pyannoteai.sdk import Client
import cv2
from scenedetect import detect, ContentDetector

print("📦 Librerías importadas")

In [ ]:
# @title
# ==========================
# Configuración de credenciales de Google Drive
# ==========================
CLIENT_ID = os.environ.get("CLIENT_ID", "")  # credencial externa
CLIENT_SECRET = os.environ.get("CLIENT_SECRET", "")  # credencial externa
REFRESH_TOKEN = os.environ.get("REFRESH_TOKEN", "")  # credencial externa

# Carpeta principal donde se guardarán todos los proyectos
MAIN_FOLDER_ID = '1YkyVktHPFDbKiWADf3KJPnkJPHNz-1uO'

print("🔑 Configuración de Drive cargada")

In [ ]:
# @title
# Configurar token de pyannote.ai de forma segura
API_TOKEN = os.environ.get("API_TOKEN", "")  # credencial externa
print("✅ Token de pyannote.ai configurado")

In [ ]:
# @title
# ==========================
# Funciones de Google Drive
# ==========================

def _build_drive_service():
    """Construye el servicio de Google Drive con las credenciales configuradas."""
    creds = Credentials(
        None,
        refresh_token=REFRESH_TOKEN,
        client_id=CLIENT_ID,
        client_secret=CLIENT_SECRET,
        token_uri='https://oauth2.googleapis.com/token'
    )
    return build('drive', 'v3', credentials=creds)

def _extract_wav(input_path: str, output_path: str):
    """Extrae audio WAV de un video usando ffmpeg."""
    subprocess.run([
        'ffmpeg', '-y', '-i', input_path,
        '-vn', '-acodec', 'pcm_s16le',
        '-ar', '44100', '-ac', '2',
        output_path
    ], check=True)

def _create_folder_in_drive(service, folder_name: str, parent_folder_id: str):
    """Crea una carpeta en Google Drive y devuelve su ID."""
    try:
        # Verificar si la carpeta ya existe
        query = f"name='{folder_name}' and parents in '{parent_folder_id}' and mimeType='application/vnd.google-apps.folder' and trashed=false"
        existing = service.files().list(q=query, fields="files(id, name)").execute()

        if existing.get('files'):
            folder_id = existing['files'][0]['id']
            print(f"📁 Carpeta '{folder_name}' ya existe (ID: {folder_id})")
            return folder_id

        # Crear nueva carpeta
        metadata = {
            'name': folder_name,
            'parents': [parent_folder_id],
            'mimeType': 'application/vnd.google-apps.folder'
        }
        folder = service.files().create(body=metadata, fields='id').execute()
        folder_id = folder.get('id')
        print(f"📁 Carpeta '{folder_name}' creada (ID: {folder_id})")
        return folder_id
    except Exception as e:
        print(f"❌ Error creando carpeta '{folder_name}': {e}")
        return None

def create_project_folders(project_name: str):
    """Crea la estructura de carpetas para un proyecto."""
    service = _build_drive_service()

    print(f"📁 Creando estructura de carpetas para proyecto: {project_name}")

    # Crear carpeta principal del proyecto
    project_folder_id = _create_folder_in_drive(service, project_name, MAIN_FOLDER_ID)
    if not project_folder_id:
        raise Exception(f"No se pudo crear la carpeta del proyecto: {project_name}")

    # Crear subcarpetas
    subcarpetas = {
        'audio': 'Audio',
        'video': 'Video',
        'diarization_results': 'Diarization Results',
        'segments_mp4': 'Segments MP4'
    }

    folder_ids = {'project': project_folder_id}

    for key, folder_name in subcarpetas.items():
        folder_id = _create_folder_in_drive(service, folder_name, project_folder_id)
        if folder_id:
            folder_ids[key] = folder_id
        else:
            print(f"⚠️ No se pudo crear la subcarpeta: {folder_name}")

    print(f"✅ Estructura de carpetas creada para '{project_name}'")
    return folder_ids

def _upload_to_drive(service, local_path: str, drive_name: str, parent_folder_id: str):
    """Sube un archivo a Google Drive en la carpeta especificada."""
    media = MediaFileUpload(local_path)
    metadata = {'name': drive_name, 'parents': [parent_folder_id]}
    file_obj = service.files().create(body=metadata, media_body=media, fields='id,webContentLink').execute()
    # Cerrar file descriptors si existen
    if hasattr(media, '_fd') and getattr(media._fd, 'closed', True) is False:
        media._fd.close()
    return file_obj

def upload_many_to_drive(file_paths, parent_folder_id):
    """Sube múltiples archivos a Drive."""
    service = _build_drive_service()
    uploaded_files = []

    for file_path in file_paths:
        if os.path.exists(file_path):
            try:
                file_name = os.path.basename(file_path)
                result = _upload_to_drive(service, file_path, file_name, parent_folder_id)
                uploaded_files.append({
                    'name': file_name,
                    'id': result.get('id'),
                    'url': result.get('webContentLink')
                })
                print(f"📤 Subido: {file_name}")
            except Exception as e:
                print(f"❌ Error subiendo {file_path}: {e}")
        else:
            print(f"⚠️ Archivo no encontrado: {file_path}")

    return uploaded_files

print("🔧 Funciones de Drive definidas")

In [ ]:
# @title
# ==========================
# Funciones de procesamiento de audio y video
# ==========================

def get_video_stem(video_path):
    """Obtiene el nombre base del video sin extensión, seguro para archivos."""
    base = os.path.basename(video_path)
    stem, _ = os.path.splitext(base)
    # Limpieza ligera para nombres
    safe = "".join(c if c.isalnum() or c in "._- " else "_" for c in stem).strip()
    return safe or "video"

def extract_audio_moviepy(video_path, audio_path):
    """Extrae audio del video usando MoviePy (formato WAV PCM 16-bit)."""
    video = VideoFileClip(video_path)
    video.audio.write_audiofile(audio_path, codec='pcm_s16le')
    video.close()
    print(f"🎵 Audio extraído: {audio_path}")

def diarize_with_sdk(audio_path, api_token):
    """Realiza diarización usando el SDK oficial de pyannote.ai."""
    client = Client(api_token)
    print("📤 Subiendo audio a pyannote.ai...")
    media_url = client.upload(audio_path)
    job_id = client.diarize(media_url)
    print("⏳ Esperando resultado de diarización...")

    while True:
        job = client.retrieve(job_id)
        status = job.get("status")
        if status == "succeeded":
            print("✅ ¡Diarización completada!")
            return job["output"]["diarization"]
        elif status in ["failed", "canceled"]:
            raise Exception(f"❌ Job {status}")
        print(f"⏳ Estado: {status}")
        time.sleep(5)

def save_json(data, filename):
    """Guarda datos en formato JSON."""
    with open(filename, "w", encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"💾 Guardado: {filename}")

def split_speakers(audio_path, diarization_json, stem_prefix):
    """Separa el audio por hablantes."""
    audio = AudioSegment.from_wav(audio_path)
    speakers = {}

    for segment in diarization_json:
        speaker = segment["speaker"]
        start = int(segment["start"] * 1000)
        end = int(segment["end"] * 1000)
        if speaker not in speakers:
            speakers[speaker] = AudioSegment.empty()
        speakers[speaker] += audio[start:end]

    out_files = []
    for speaker, seg_audio in speakers.items():
        out_path = f"{stem_prefix}_speaker_{speaker}.wav"
        seg_audio.export(out_path, format="wav")
        out_files.append(out_path)
        print(f"🎤 Audio de {speaker}: {out_path}")

    return out_files

print("🎬 Funciones de procesamiento definidas")

In [ ]:
# @title
# ==========================
# Funciones de análisis de segmentos silenciosos
# ==========================

def find_silent_segments(diarization_json, audio_duration=None):
    """Encuentra segmentos sin hablantes (silencios)."""
    segments = sorted(diarization_json, key=lambda x: x["start"])
    if not segments:
        return []

    silent_segments = []
    segment_id = 1

    # Silencio inicial
    if segments[0]["start"] > 0:
        silent_segments.append({
            "id": segment_id,
            "start": 0.0,
            "end": segments[0]["start"],
            "duration": round(segments[0]["start"] - 0.0, 3)
        })
        segment_id += 1

    # Silencios entre segmentos
    for i in range(len(segments) - 1):
        current_end = segments[i]["end"]
        next_start = segments[i + 1]["start"]
        if next_start > current_end:
            silent_segments.append({
                "id": segment_id,
                "start": current_end,
                "end": next_start,
                "duration": round(next_start - current_end, 3)
            })
            segment_id += 1

    # Silencio final
    if audio_duration and segments[-1]["end"] < audio_duration:
        silent_segments.append({
            "id": segment_id,
            "start": segments[-1]["end"],
            "end": audio_duration,
            "duration": round(audio_duration - segments[-1]["end"], 3)
        })

    return silent_segments

def join_close_segments(silent_segments, video_path, max_gap=2.0):
    """Une segmentos cercanos verificando cambios de escena."""
    print(f"🔗 Uniendo segmentos con gap <= {max_gap} segundos...")
    scenes = detect(video_path, ContentDetector(threshold=27.0))
    scene_times = [(scene[0].get_seconds(), scene[1].get_seconds()) for scene in scenes]

    joined_segments = []
    i = 0

    while i < len(silent_segments):
        current_segment = silent_segments[i].copy()
        segments_to_join = [current_segment]
        j = i + 1

        while j < len(silent_segments):
            gap = silent_segments[j]["start"] - current_segment["end"]
            if gap <= max_gap:
                segments_to_join.append(silent_segments[j].copy())
                current_segment = silent_segments[j]
                j += 1
            else:
                break

        if len(segments_to_join) == 1:
            joined_segments.append(segments_to_join[0])
            i += 1
        else:
            joined_start = segments_to_join[0]["start"]
            joined_end = segments_to_join[-1]["end"]
            joined_duration = joined_end - joined_start

            # Verificar cambios de escena
            has_scene_change = False
            for scene_start, scene_end in scene_times:
                if (scene_start > joined_start and scene_start < joined_end):
                    has_scene_change = True
                    break

            if has_scene_change:
                print(f"🎬 Cambio de escena detectado en {joined_start:.3f}s - {joined_end:.3f}s")
                joined_segments.extend(segments_to_join)
            else:
                joined_segment = {
                    "id": len(joined_segments) + 1,
                    "start": joined_start,
                    "end": joined_end,
                    "duration": round(joined_duration, 3),
                    "joined_from": [seg["id"] for seg in segments_to_join]
                }
                print(f"🔗 Uniendo {joined_start:.3f}s - {joined_end:.3f}s")
                joined_segments.append(joined_segment)
            i = j

    # Reasignar IDs
    for idx, segment in enumerate(joined_segments, start=1):
        segment["id"] = idx

    return joined_segments

def detect_scene_changes_in_segments(video_path, silent_segments, min_scene_duration=2.0):
    """Detecta cambios de escena dentro de segmentos silenciosos."""
    print("🎬 Detectando cambios de escena...")
    scenes = detect(video_path, ContentDetector(threshold=27.0))
    scene_times = [(scene[0].get_seconds(), scene[1].get_seconds()) for scene in scenes]
    print(f"📊 Detectadas {len(scene_times)} escenas")

    processed_segments = []

    for segment in silent_segments:
        segment_start = segment["start"]
        segment_end = segment["end"]
        overlapping_scenes = []

        for scene_start, scene_end in scene_times:
            if (scene_start < segment_end and scene_end > segment_start):
                overlap_start = max(scene_start, segment_start)
                overlap_end = min(scene_end, segment_end)
                if overlap_end > overlap_start:
                    overlapping_scenes.append((overlap_start, overlap_end))

        if len(overlapping_scenes) <= 1:
            processed_segments.append(segment)
        else:
            sub_segments = []
            current_id = segment["id"]

            for i, (scene_start, scene_end) in enumerate(overlapping_scenes):
                sub_duration = scene_end - scene_start
                if sub_duration >= min_scene_duration:
                    sub_segments.append({
                        "id": f"{current_id}_{i+1}",
                        "start": round(scene_start, 3),
                        "end": round(scene_end, 3),
                        "duration": round(sub_duration, 3),
                        "original_segment_id": current_id,
                        "scene_number": i + 1
                    })

            if not sub_segments:
                processed_segments.append(segment)
            else:
                processed_segments.extend(sub_segments)

    # Reasignar IDs
    for idx, segment in enumerate(processed_segments, start=1):
        segment["id"] = idx

    return processed_segments

print("🔍 Funciones de análisis definidas")

## 📋 Descripción del Pipeline

Este notebook ejecuta un pipeline completo de análisis de video que incluye:

### 🔄 Proceso Automático:
1. **📤 Subida de Video**: Sube el video a Google Drive
2. **🎵 Extracción de Audio**: Extrae audio WAV y lo sube a Drive
3. **🎤 Diarización**: Identifica y separa hablantes usando pyannote.ai
4. **👥 Separación por Hablante**: Genera archivos de audio individuales
5. **🔇 Análisis de Silencios**: Detecta segmentos sin habla
6. **🎬 Detección de Escenas**: Identifica cambios de escena en silencios
7. **📊 Generación de Reportes**: Crea archivos JSON con análisis detallado
8. **☁️ Subida a Drive**: Guarda todos los resultados automáticamente

### 📁 Archivos Generados:
- `{video}_audio.wav` - Audio extraído
- `{video}_diarization.json` - Resultado de diarización
- `{video}_speaker_X.wav` - Audio por cada hablante
- `{video}_silent_segments_joined.json` - Segmentos silenciosos unidos
- `{video}_silent_segments_with_scenes.json` - Análisis final con escenas

### 🔧 Configuración:
- **Umbral de silencio**: 2 segundos mínimo
- **Gap máximo para unir**: 2 segundos
- **Duración mínima de escena**: 2 segundos
- **Umbral de detección de escena**: 27.0

Todos los archivos se nombran automáticamente basándose en el nombre del video original y se organizan en las carpetas correspondientes de Google Drive.

In [ ]:
# @title
# ==========================
# Funciones para recortar y subir segmentos de video
# ==========================

def cut_video_segment(video_path: str, output_path: str, start_time: float, end_time: float):
    """Recorta un segmento de video usando ffmpeg."""
    try:
        # Usar stream copy para no re-encode si es posible, o re-encode si es necesario
        subprocess.run([
            'ffmpeg', '-y', '-i', video_path, '-ss', str(start_time), '-to', str(end_time),
            '-c', 'copy', output_path
        ], check=True, capture_output=True, text=True)
        print(f"🎬 Segmento recortado: {output_path} ({start_time:.3f}s - {end_time:.3f}s)")
    except subprocess.CalledProcessError as e:
        # Fallback: Re-encode si stream copy falla
        print(f"⚠️ Falló stream copy para {output_path}, intentando re-encode: {e.stderr}")
        try:
            subprocess.run([
                'ffmpeg', '-y', '-i', video_path, '-ss', str(start_time), '-to', str(end_time),
                '-c:v', 'libx264', '-c:a', 'aac', '-strict', 'experimental', output_path
            ], check=True, capture_output=True, text=True)
            print(f"✅ Segmento re-encoded y guardado: {output_path}")
        except subprocess.CalledProcessError as e2:
            print(f"❌ Falló re-encode para {output_path}: {e2.stderr}")
            raise e2


def process_and_upload_segments(video_path: str, segments: list, segments_folder_id: str):
    """Recorta los segmentos de video especificados y los sube a Google Drive."""
    service = _build_drive_service()
    uploaded_segments = []
    temp_segment_files = []

    os.makedirs('segments_temp', exist_ok=True)

    print("\n🎬 Recortando y subiendo segmentos de video...")

    for segment in segments:
        segment_id = segment.get("id", "unknown")
        start_time = segment["start"]
        end_time = segment["end"]
        # Asegurarse de que el nombre del archivo sea seguro y descriptivo
        output_name = f"segmento_{segment_id}.mp4"
        output_path = os.path.join('segments_temp', output_name)

        try:
            cut_video_segment(video_path, output_path, start_time, end_time)
            temp_segment_files.append(output_path)

            # Subir el segmento recortado a la carpeta de segmentos MP4
            print(f"📤 Subiendo segmento {segment_id} a Google Drive...")
            result = _upload_to_drive(service, output_path, output_name, segments_folder_id)
            uploaded_segments.append({
                'id': segment_id,
                'name': output_name,
                'drive_id': result.get('id'),
                'url': result.get('webContentLink')
            })
            print(f"✅ Segmento {segment_id} subido - ID: {result.get('id')}")

        except Exception as e:
            print(f"❌ Error procesando/subiendo segmento {segment_id}: {e}")

    # Limpiar archivos temporales de segmentos
    print('🧹 Limpiando archivos temporales de segmentos...')
    time.sleep(0.5)
    for temp_file in temp_segment_files:
        try:
            if os.path.exists(temp_file):
                os.remove(temp_file)
        except Exception as e:
            print(f"⚠️ No se pudo eliminar {temp_file}: {e}")

    try:
        # Eliminar la carpeta temporal si está vacía
        if os.path.exists('segments_temp') and not os.listdir('segments_temp'):
            os.rmdir('segments_temp')
    except Exception as e:
         print(f"⚠️ No se pudo eliminar la carpeta temporal 'segments_temp': {e}")


    return uploaded_segments

print("✂️ Funciones de recorte y subida de segmentos definidas")

In [ ]:
# @title
# ==========================
# Función principal de subida y procesamiento
# ==========================

def upload_and_process_video():
    """Sube un video, extrae WAV, sube ambos a Drive y ejecuta el pipeline de diarización."""

    # Solicitar nombre del proyecto
    project_name = input('� Ingresa el nombre del proyecto: ').strip()
    if not project_name:
        print('❌ Debes ingresar un nombre para el proyecto')
        return None

    print(f'🎯 Proyecto: {project_name}')

    # Crear estructura de carpetas en Drive
    try:
        folder_ids = create_project_folders(project_name)
        print('✅ Estructura de carpetas creada exitosamente')
    except Exception as e:
        print(f'❌ Error creando carpetas: {e}')
        return None

    print('�📁 Selecciona el archivo de video para subir:')
    uploaded = files.upload()
    if not uploaded:
        print('❌ No se seleccionó ningún archivo')
        return None

    original_name = list(uploaded.keys())[0]
    base = str(Path(original_name).stem)
    wav_name = base + '.wav'

    os.makedirs('temp', exist_ok=True)
    video_path = os.path.join('temp', original_name)
    # Mover el archivo subido a temp conservando el nombre
    shutil.move(original_name, video_path)
    print(f'📹 Video guardado: {video_path}')

    try:
        print('🎵 Extrayendo audio (WAV)...')
        wav_path = os.path.join('temp', wav_name)
        _extract_wav(video_path, wav_path)
        print(f'✅ Audio extraído: {wav_path}')

        print('🔐 Autenticando con Google Drive...')
        service = _build_drive_service()

        print('📤 Subiendo video a carpeta Video...')
        video_upload = _upload_to_drive(service, video_path, original_name, folder_ids['video'])
        print(f"✅ Video subido - ID: {video_upload.get('id')}")

        print('📤 Subiendo audio a carpeta Audio...')
        audio_upload = _upload_to_drive(service, wav_path, wav_name, folder_ids['audio'])
        print(f"✅ Audio subido - ID: {audio_upload.get('id')}")

        # ===== INICIO DEL PIPELINE DE DIARIZACIÓN =====
        print('\n🚀 === INICIANDO PIPELINE DE DIARIZACIÓN ===')

        # Preparar nombres de archivos basados en el video
        video_stem = get_video_stem(video_path)
        audio_file_diarization = f"{video_stem}_audio.wav"
        diarization_json_file = f"{video_stem}_diarization.json"
        silent_joined_file = f"{video_stem}_silent_segments_joined.json"
        silent_with_scenes_file = f"{video_stem}_silent_segments_with_scenes.json"

        # Copiar el audio WAV para el procesamiento
        shutil.copy2(wav_path, audio_file_diarization)

        # Paso 1: Diarización con pyannote.ai
        print('🎤 Realizando diarización...')
        diarization = diarize_with_sdk(audio_file_diarization, API_TOKEN)
        save_json(diarization, diarization_json_file)

        # Paso 2: Separar audios por hablante
        print('👥 Separando audios por hablante...')
        speaker_files = split_speakers(audio_file_diarization, diarization, video_stem)

        # Paso 3: Análisis de segmentos silenciosos
        print('🔇 Analizando segmentos silenciosos...')
        audio_duration = AudioSegment.from_wav(audio_file_diarization).duration_seconds
        silent_segments = find_silent_segments(diarization, audio_duration)

        # Filtrar silencios > 2s
        long_silent_segments = [s for s in silent_segments if s["duration"] > 2.0]
        for idx, segment in enumerate(long_silent_segments, start=1):
            segment["id"] = idx

        print(f"📊 Encontrados {len(long_silent_segments)} segmentos silenciosos > 2s")

        # Paso 4: Unir segmentos cercanos
        print('🔗 Uniendo segmentos cercanos...')
        joined_segments = join_close_segments(long_silent_segments, video_path, max_gap=2.0)
        save_json(joined_segments, silent_joined_file)
        print(f"📊 {len(joined_segments)} segmentos después de unir")

        # Paso 5: Detectar cambios de escena
        print('🎬 Detectando cambios de escena en segmentos...')
        final_segments = detect_scene_changes_in_segments(video_path, joined_segments, min_scene_duration=2.0)
        save_json(final_segments, silent_with_scenes_file)

        # Mostrar resumen de segmentos finales
        print(f"\n📊 === RESUMEN FINAL: {len(final_segments)} segmentos ===")
        for segment in final_segments[:10]:  # Mostrar solo los primeros 10
            if "scene_number" in segment:
                print(f"  🎬 Segmento {segment['id']}: {segment['start']:.3f}s - {segment['end']:.3f}s (Escena {segment['scene_number']})")
            else:
                print(f"  🔇 Segmento {segment['id']}: {segment['start']:.3f}s - {segment['end']:.3f}s")

        if len(final_segments) > 10:
            print(f"  ... y {len(final_segments) - 10} segmentos más")

        # === Paso: Recortar y subir segmentos de video ===
        print('\n✂️ === RECORTANDO Y SUBIENDO SEGMENTOS DE VIDEO ===')
        uploaded_video_segments = process_and_upload_segments(video_path, final_segments, folder_ids['segments_mp4'])
        print(f"✅ {len(uploaded_video_segments)} segmentos de video subidos a carpeta Segments MP4")

        # Paso 6: Subir audios separados por hablante a carpeta Audio
        print('\n📤 Subiendo audios de hablantes a carpeta Audio...')
        uploaded_speaker_audios = upload_many_to_drive(speaker_files, folder_ids['audio'])

        # Paso 7: Subir archivos de diarización y análisis a carpeta Diarization Results
        print('\n📤 Subiendo resultados de diarización...')
        diarization_files = [
            audio_file_diarization,
            diarization_json_file,
            silent_joined_file,
            silent_with_scenes_file
        ]
        uploaded_diarization_results = upload_many_to_drive(diarization_files, folder_ids['diarization_results'])

        # Limpieza de archivos temporales
        print('🧹 Limpiando archivos temporales...')
        time.sleep(0.5)
        temp_files = [video_path, wav_path, audio_file_diarization] + diarization_files + speaker_files
        for temp_file in temp_files:
            try:
                if os.path.exists(temp_file):
                    os.remove(temp_file)
            except Exception as e:
                print(f"⚠️ No se pudo eliminar {temp_file}: {e}")

        # Resultado final
        result = {
            'project_name': project_name,
            'folder_ids': folder_ids,
            'video': {
                'name': original_name,
                'file_id': video_upload.get('id'),
                'download_url': video_upload.get('webContentLink')
            },
            'audio': {
                'name': wav_name,
                'file_id': audio_upload.get('id'),
                'download_url': audio_upload.get('webContentLink')
            },
            'speaker_audios': uploaded_speaker_audios,
            'diarization_results': uploaded_diarization_results,
            'video_segments': uploaded_video_segments,
            'segments_summary': {
                'total_segments': len(final_segments),
                'segments_with_scenes': len([s for s in final_segments if 'scene_number' in s]),
                'speakers_found': len(speaker_files)
            }
        }

        print('\n🎉 === PROCESO COMPLETADO EXITOSAMENTE ===')
        print(f"🎯 Proyecto: {project_name}")
        print(f"📁 Carpeta principal ID: {folder_ids['project']}")
        print(f"📹 Video original: {result['video']['download_url']}")
        print(f"🎵 Audio original: {result['audio']['download_url']}")
        print(f"� Audios de hablantes: {len(result['speaker_audios'])} archivos en carpeta Audio")
        print(f"📊 Resultados de diarización: {len(result['diarization_results'])} archivos en carpeta Diarization Results")
        print(f"🎬 Segmentos de video: {len(result['video_segments'])} archivos en carpeta Segments MP4")
        print(f"👥 Hablantes encontrados: {result['segments_summary']['speakers_found']}")
        print(f"🎬 Segmentos totales analizados: {result['segments_summary']['total_segments']}")

        return result

    except subprocess.CalledProcessError as ffmpeg_error:
        print(f'❌ Error con ffmpeg: {ffmpeg_error}')
        return None
    except Exception as e:
        print(f'❌ Error general: {e}')
        import traceback
        traceback.print_exc()
        return None

print('✅ === SISTEMA LISTO. Ejecuta upload_and_process_video() para iniciar ===')
print('📁 Estructura de carpetas que se creará:')
print('   📂 [Nombre del Proyecto]')
print('      ├── 📂 Audio (audio original + audios por hablante)')
print('      ├── 📂 Video (video original)')
print('      ├── 📂 Diarization Results (JSONs y audio procesado)')
print('      └── 📂 Segments MP4 (segmentos recortados)')

In [ ]:
# @title
# ==========================
# EJECUTAR EL PIPELINE COMPLETO
# ==========================

# Ejecuta esta celda para iniciar todo el proceso:
# 1. Solicitar nombre del proyecto
# 2. Crear estructura de carpetas organizadas en Drive
# 3. Subir video y audio a carpetas correspondientes
# 4. Realizar diarización completa
# 5. Generar análisis de segmentos
# 6. Subir todos los resultados organizadamente

result = upload_and_process_video()

if result:
    print("\n📋 === RESUMEN FINAL ===")
    print(f"🎯 Proyecto: {result['project_name']}")
    print(f"📁 Estructura creada en Drive:")
    print(f"   📂 {result['project_name']} (ID: {result['folder_ids']['project']})")
    print(f"      ├── 📂 Audio (ID: {result['folder_ids']['audio']})")
    print(f"      ├── 📂 Video (ID: {result['folder_ids']['video']})")
    print(f"      ├── 📂 Diarization Results (ID: {result['folder_ids']['diarization_results']})")
    print(f"      └── 📂 Segments MP4 (ID: {result['folder_ids']['segments_mp4']})")
    print(f"\n📊 Archivos procesados:")
    print(f"   📹 Video original: {result['video']['name']}")
    print(f"   🎵 Audio original: {result['audio']['name']}")
    print(f"   👥 Audios de hablantes: {len(result['speaker_audios'])} archivos")
    print(f"   📊 Archivos de análisis: {len(result['diarization_results'])} archivos")
    print(f"   🎬 Segmentos de video: {len(result['video_segments'])} archivos")
    print(f"\n🔢 Estadísticas:")
    print(f"   👥 Hablantes detectados: {result['segments_summary']['speakers_found']}")
    print(f"   🎬 Segmentos analizados: {result['segments_summary']['total_segments']}")
    print(f"   🎭 Segmentos con escenas: {result['segments_summary']['segments_with_scenes']}")
    print(f"\n🔗 Enlaces principales:")
    print(f"   📹 Video: {result['video']['download_url']}")
    print(f"   🎵 Audio: {result['audio']['download_url']}")
else:
    print("❌ El procesamiento falló. Revisa los errores anteriores.")

# Documentación del flujo de procesamiento de videos con audiodescripción
Este notebook automatiza el procesamiento de videos para generar audiodescripciones accesibles para personas ciegas o con baja visión. El flujo general es:

1. **Carga de videos**: Los videos se toman de la carpeta `videos` (puede estar sincronizada con Google Drive).
2. **Extracción de keyframes**: Se seleccionan los frames más representativos usando el modelo CLIP.
3. **Generación de audiodescripciones**: Se utiliza la API de OpenAI para describir cada keyframe siguiendo pautas de accesibilidad y prosodia.
4. **Análisis de prosodia**: Se analiza el tono emocional y prosódico global del video.
5. **Resumen del video**: Se genera un resumen narrativo ajustado a la emoción y duración del video.
6. **Organización de resultados**: Los resultados se guardan en subcarpetas dentro de la carpeta de salida, una por cada video.

**Dependencias principales:**
- OpenAI (API)
- Katna
- scenedetect
- torch, clip, numpy
- PIL, opencv-python
- Google Drive API (opcional para integración)

---

**Notas:**
- El código está modularizado y documentado por bloques.
- Se recomienda ejecutar primero el bloque de instalación de dependencias.
- Es necesario tener una API Key válida de OpenAI para la generación de descripciones y resúmenes.


In [ ]:
import numpy;
print(numpy.__version__)

In [ ]:
# Re-install NumPy to a compatible version for libraries like OpenCV
!pip install numpy==1.23.5 --force-reinstall

print('✅ NumPy re-installed to 1.23.5')

In [ ]:
# @title
# Instalación de dependencias necesarias para el procesamiento de videos y audiodescripción
!pip install openai katna scenedetect torch ftfy regex tqdm pillow git+https://github.com/openai/CLIP.git --quiet

# Si necesitas integración con Google Drive, ejecuta también:
!pip install --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib --quiet

In [ ]:
# @title
# ==========================
# Importación de librerías
# ==========================
import cv2
import os
import json
import datetime
from PIL import Image
from Katna.video import Video
from Katna.writer import KeyFrameDiskWriter
from scenedetect import VideoManager, SceneManager
from scenedetect.detectors import ContentDetector
import openai
import base64
import torch
import clip
import numpy as np
import time
import re

# ==========================
# Configuración de credenciales de Google Drive
# ==========================
CLIENT_ID = os.environ.get("CLIENT_ID", "")  # credencial externa
CLIENT_SECRET = os.environ.get("CLIENT_SECRET", "")  # credencial externa
REFRESH_TOKEN = os.environ.get("REFRESH_TOKEN", "")  # credencial externa

# ID del folder input (se solicitará al usuario)
FOLDER_INPUT = None  # Se definirá dinámicamente
FOLDER_SEGMENTS_DESCRIPTIONS = None  # Se creará dinámicamente

print('🔑 Configuración de Drive cargada')

# ==========================
# Función: Solicitar ID de carpeta input y crear carpeta de destino
# ==========================
def setup_folders():
    """
    Solicita al usuario el ID del folder input y crea la carpeta 'segments descriptions'
    al mismo nivel.

    Returns:
        tuple: (folder_input_id, folder_segments_descriptions_id)
    """
    global FOLDER_INPUT, FOLDER_SEGMENTS_DESCRIPTIONS

    # Solicitar ID del folder input
    folder_input_id = input("📁 Por favor, ingresa el ID del folder input: ").strip()

    if not folder_input_id:
        raise ValueError("❌ El ID del folder input es requerido")

    FOLDER_INPUT = folder_input_id

    # Conectar a Drive para crear la nueva carpeta
    try:
        service = connect_to_drive()

        # Obtener información del folder input para determinar su carpeta padre
        input_folder_info = service.files().get(fileId=folder_input_id, fields='parents,name').execute()
        parent_id = input_folder_info['parents'][0] if input_folder_info.get('parents') else None

        if not parent_id:
            raise ValueError("❌ No se pudo determinar la carpeta padre del folder input")

        # Crear carpeta 'segments descriptions' al mismo nivel
        segments_folder_name = 'segments descriptions'

        # Verificar si ya existe una carpeta con ese nombre
        existing_folders = service.files().list(
            q=f"name='{segments_folder_name}' and parents in '{parent_id}' and mimeType='application/vnd.google-apps.folder'"
        ).execute()

        if existing_folders.get('files'):
            # Ya existe, usar la existente
            segments_folder_id = existing_folders['files'][0]['id']
            print(f"📁 Usando carpeta existente 'segments descriptions': {segments_folder_id}")
        else:
            # Crear nueva carpeta
            folder_metadata = {
                'name': segments_folder_name,
                'mimeType': 'application/vnd.google-apps.folder',
                'parents': [parent_id]
            }

            created_folder = service.files().create(body=folder_metadata, fields='id').execute()
            segments_folder_id = created_folder.get('id')
            print(f"✅ Carpeta 'segments descriptions' creada exitosamente: {segments_folder_id}")

        FOLDER_SEGMENTS_DESCRIPTIONS = segments_folder_id

        return folder_input_id, segments_folder_id

    except Exception as e:
        raise Exception(f"❌ Error configurando carpetas: {str(e)}")

# ==========================
# Función: Filtrar archivos que contengan "segmento"
# ==========================
def filter_segment_files(file_list, filter_term="segmento"):
    """
    Filtra una lista de archivos para incluir solo aquellos que contengan el término especificado.

    Args:
        file_list: Lista de nombres de archivos o rutas
        filter_term: Término que debe estar presente en el nombre del archivo (por defecto: "segmento")

    Returns:
        Lista filtrada de archivos
    """
    filtered_files = []
    for file_path in file_list:
        filename = os.path.basename(file_path) if isinstance(file_path, str) and os.path.sep in file_path else file_path
        if filter_term.lower() in filename.lower():
            filtered_files.append(file_path)

    print(f"📁 Filtrado aplicado: {len(filtered_files)} de {len(file_list)} archivos contienen '{filter_term}'")
    if filtered_files:
        print("📝 Archivos seleccionados:")
        for file_path in filtered_files:
            filename = os.path.basename(file_path) if isinstance(file_path, str) and os.path.sep in file_path else file_path
            print(f"  - {filename}")

    return filtered_files

# ==========================
# Función: Validación de resumen
# ==========================
def is_valid_summary(summary_text):
    """
    Verifica si el resumen es válido (no está vacío, no son solo guiones, etc.)
    """
    if not summary_text or not summary_text.strip():
        return False
    cleaned_text = summary_text.strip()
    if re.match(r'^[\s\-_*]+$', cleaned_text):
        return False
    if len(cleaned_text) < 10:
        return False
    if not re.search(r'[a-zA-ZáéíóúñÁÉÍÓÚÑ]', cleaned_text):
        return False
    error_patterns = [
        'error', 'no se pudo', 'no se puede', 'imposible', 'falló', 'failed', 'empty', 'vacío', 'sin contenido'
    ]
    cleaned_lower = cleaned_text.lower()
    for pattern in error_patterns:
        if pattern in cleaned_lower:
            return False
    return True

# ==========================
# Función: Extracción de keyframes usando CLIP (ajustable)
# ==========================
def extract_keyframes_clip_adjustable(video_path, output_folder='keyframes_clip',
                                    diversity_threshold=0.7,
                                    similarity_threshold=0.8,
                                    min_keyframes=5,
                                    max_keyframes=15,
                                    frame_sampling_rate=30):
    """
    Extrae keyframes usando CLIP con umbrales ajustables
    """
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model, preprocess = clip.load('ViT-B/32', device=device)
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_rate = cap.get(cv2.CAP_PROP_FPS)
    if frame_rate == 0:
        print('Error: No se pudo obtener la tasa de cuadros del video.')
        return [], 0
    video_duration = total_frames / frame_rate
    frame_interval = max(1, frame_sampling_rate)
    frames = []
    frame_indices = []
    print(f'Analizando video con {total_frames} frames, muestreando cada {frame_interval} frames...')
    for i in range(0, total_frames, frame_interval):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = cap.read()
        if not ret:
            continue
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image = preprocess(Image.fromarray(frame_rgb)).unsqueeze(0).to(device)
        frames.append((image, frame_rgb, i))
        frame_indices.append(i)
    cap.release()
    if not frames:
        return [], video_duration
    frames_tensor = torch.cat([f[0] for f in frames])
    with torch.no_grad():
        image_features = model.encode_image(frames_tensor)
    image_features /= image_features.norm(dim=-1, keepdim=True)
    similarities = torch.mm(image_features, image_features.T)
    diversity_scores = similarities.sum(dim=1)
    diversity_scores = (diversity_scores - diversity_scores.min()) / (diversity_scores.max() - diversity_scores.min())
    if len(frames) <= min_keyframes:
        target_frames = len(frames)
    else:
        target_frames = min(max_keyframes, max(min_keyframes, len(frames) // 2))
    print(f'Frames muestreados: {len(frames)}, Target frames: {target_frames}')
    diverse_indices = torch.argsort(diversity_scores)[:target_frames]
    final_keyframes = diverse_indices.tolist()
    if len(final_keyframes) < min_keyframes and len(frames) >= min_keyframes:
        additional_needed = min_keyframes - len(final_keyframes)
        remaining_indices = [i for i in range(len(frames)) if i not in final_keyframes]
        if remaining_indices:
            remaining_diversity = diversity_scores[remaining_indices]
            best_remaining = torch.argsort(remaining_diversity)[:additional_needed]
            for idx in best_remaining:
                final_keyframes.append(remaining_indices[idx.item()])
    final_keyframes.sort()
    selected_frames = [frames[idx] for idx in final_keyframes]
    os.makedirs(output_folder, exist_ok=True)
    saved_frames = []
    keyframes_json = []
    for idx, (_, frame, frame_num) in enumerate(selected_frames):
        frame_path = os.path.join(output_folder, f'keyframe_{frame_num}.jpg')
        Image.fromarray(frame).save(frame_path, format='JPEG', quality=95)
        saved_frames.append(frame_num)
        frame_time = frame_num / frame_rate
        keyframes_json.append({
            'frame_index': idx,
            'frame_number': frame_num,
            'timestamp': frame_time,
            'filename': f'keyframe_{frame_num}.jpg'
        })
        print(f'Keyframe guardado: keyframe_{frame_num}.jpg')
    json_path = os.path.join(output_folder, 'keyframes.json')
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(keyframes_json, f, ensure_ascii=False, indent=2)
    print(f'Total keyframes extraídos con CLIP: {len(saved_frames)}')
    print(f'Configuración usada - Diversidad: {diversity_threshold}, Similitud: {similarity_threshold}')
    return saved_frames, video_duration

# ==========================
# Función: Generar audiodescripciones con OpenAI
# ==========================
def generate_audiodescriptions(api_key, image_folder, output_json="audiodescriptions.json", video_duration=None):
    client = openai.OpenAI(api_key=api_key)
    if video_duration is None:
        max_chars = 30
    else:
        max_chars = int(video_duration * 30)
    descriptions = []
    total_tokens = 0
    prompt = (
        f"Rol:Eres un Especialista en Audiodescripción para personas ciegas o con baja visión.\n"
        f"Tarea: Genera una descripción de la imagen y generar prosodia con base al modelado prosodico (Intensidad, Tono, Duración, Ritmo)\n"
        f"Elementos esenciales a cubrir:\n"
        f"- [Quién] Identifica personajes, características distintivas y acciones\n"
        f"- [Qué] Describe objetos, eventos e interacciones relevantes\n"
        f"- [Dónde] Proporciona contexto del entorno\n"
        f"- [Cuándo] Indica momento del día si es relevante\n"
        f"- [Acción] Especifica movimientos significativos\n"
        f"- [Cambios de Escena] Indica transiciones temporales importantes\n"
        f"- [Logos] Si hay logotipos, símbolos o elementos gráficos identificativos, descríbelos detalladamente, incluyendo su forma, colores predominantes, ubicación en la imagen y cualquier elemento visual relevante. Explica su posible función o significado en la escena.\n\n"

        f"Mecanismos de Atención (prioriza los más relevantes según el contenido):\n"
        f" 1. Personajes:\n"
        f" Definición conceptual: Entidades narrativas con identidad propia que participan en la acción, portan rasgos físicos, psicológicos, sociales y emocionales, y cuya presencia tiene un peso semántico en la construcción del relato. Su descripción debe priorizar los atributos relevantes para la trama y los aspectos que marcan su evolución emocional. En la audiodescripción, el énfasis está en la caracterización funcional y afectiva.\n"
        f" Ejemplo: Un joven con ropa desgastada y una mirada esquiva recorre la estación de tren sin rumbo aparente. Su caracterización transmite anticipación (media: ansiedad), marcando una identidad emocionalmente suspendida entre la espera y la incertidumbre.\n"
        f" Caracteristicas:\n"
        f"   - Género y edad aproximada\n"
        f"   - Real o animado\n"
        f"   - Accesorios distintivos\n"
        f"   - Rasgos faciales destacables\n"
        f"   - Justificación: Especifica CUÁL personaje y QUÉ aspectos específicos son relevantes, explicando por qué\n"
        f" 2. Expresiones Faciales:\n"
        f" Definición conceptual: Configuraciones musculares del rostro que comunican emociones, intenciones o reacciones internas de los personajes. Son fundamentales para la lectura emocional de la escena y, desde una perspectiva semántica, funcionan como signos icónicos que conectan la interioridad del personaje con el espectador. La audiodescripción debe traducir estas expresiones en términos que evoquen la intensidad y cualidad emocional de forma precisa.\n"
        f" Ejemplo: Un pasillo estrecho, con paredes agrietadas y una luz parpadeante al fondo, acompañado por un zumbido casi imperceptible con un fuerte olor a muerte. La composición del escenario y el clima evocan una emoción de miedo (alto: terror), preparando al oyente para una posible amenaza o evento inesperado.\n"
        f" Caracteristicas:\n"
        f"   - Emociones (Alegría (Bajo: Serenidad, Alto: Éxtasis), Confianza (Bajo: Aceptación, Alto: Admiración), Miedo (Bajo: Aprensión, Alto: Terror), Sorpresa (Bajo: Distracción, Alto: Asombro), Tristeza (Bajo: Duelo, Alto: Pensamiento profundo), Disgusto (Bajo: Aburrimiento, Alto: Odio), Ira (Bajo: Molestia, Alto: Furia), Anticipación (Bajo: Interés, Alto: Vigilancia).)\n"
        f"   - Justificación: Especifica CUÁL Configuración musculares del rostro que comunican emociones y QUÉ emoción específica, explicando su impacto\n"
        f" 3. Acciones:\n"
        f" Definición conceptual: Conjunto de movimientos, gestos o comportamientos ejecutados por personajes o elementos en la escena que impulsan el desarrollo narrativo o comunican intenciones, emociones o cambios de estado. Desde la dimensión semántica, las acciones son unidades mínimas de sentido que articulan la lógica de los acontecimientos, y su mención en la audiodescripción debe enfatizar lo relevante para la comprensión narrativa y emocional del contenido.\n"
        f" Ejemplo: Una mujer abre lentamente una carta, la lee y deja caer los brazos. La acción comunica una emoción de tristeza (medio: tristeza). La pausa en el movimiento y el descenso corporal son claves para expresar el peso emocional del contenido leído.\n"
        f" Caracteristicas:\n"
        f"   - Movimientos de personajes\n"
        f"   - Movimientos de objetos relevantes\n"
        f"   - Justificación: Especifica CUÁL acción concreta y CUÁL es su importancia específica en la escena\n"
        f" 4. Escenarios, clima y atmósfera:\n"
        f" Definición conceptual: Conjunto articulado por el espacio físico representado (escenario) y la carga sensorial y afectiva que lo impregna (clima y atmósfera), que en conjunto configuran el trasfondo emocional y simbólico de la acción narrativa. Desde la dimensión semántica, esta categoría abarca tanto los elementos materiales del entorno (arquitectura, objetos, iluminación, distribución espacial) como sus cualidades expresivas (tonalidad emocional, tensión ambiental, ritmo perceptivo). En la audiodescripción, su traducción debe priorizar los aspectos sensoriales que modelan la experiencia emocional del espectador, activando emociones como miedo, alegría o tristeza, y facilitando la inmersión narrativa del oyente.\n"
        f" Ejemplo: Un pasillo estrecho, con paredes agrietadas y una luz parpadeante al fondo, acompañado por un zumbido casi imperceptible con un fuerte olor a muerte. La composición del escenario y el clima evocan una emoción de miedo (alto: terror), preparando al oyente para una posible amenaza o evento inesperado.\n"
        f" Caracteristicas:\n"
        f"   - Ubicación y ambientación\n"
        f"   - Colores predominantes\n"
        f"   - Temporalidad\n"
        f"   - Condiciones ambientales\n"
        f"   - Ambiente general\n"
        f"   - Justificación: Especifica QUÉ elementos específicos son relevantes y CUÁL es su importancia específica en la escena\n"
        f" 5. Postura y Lenguaje Corporal:\n"
        f" Definición conceptual: Configuraciones físicas del cuerpo y sus movimientos no verbales que expresan estados emocionales, disposiciones actitudinales o relaciones interpersonales. Actúan como signos visuales de significación emocional y contextual. En audiodescripción, deben ser traducidos en imágenes verbales que permitan inferir el estado emocional, la intención o la tensión narrativa de la escena.\n"
        f" Ejemplo: Una mujer se mantiene erguida, con los brazos cruzados y la cabeza levemente inclinada hacia atrás al escuchar a su interlocutor. Su postura indica rechazo (bajo: aburrimiento), marcando distancia emocional y juicio negativo.\n"
        f" Caracteristicas:\n"
        f"   - Gestos significativos\n"
        f"   - Posturas relevantes\n"
        f"   - Justificación: Especifica QUÉ gesto o postura específica y CUÁL es su significado concreto\n"
        f" 6. Interacciones:\n"
        f" Definición conceptual: Relaciones dinámicas entre personajes o entre personajes y objetos/entornos, que revelan vínculos afectivos, sociales o simbólicos. Las interacciones son claves para la comprensión de los roles, jerarquías, conflictos o alianzas en el universo narrativo. En la audiodescripción, se deben describir no solo los actos, sino también el carácter relacional y emocional de los mismos.\n"
        f" Ejemplo: Una niña se recuesta en el hombro de su abuela mientras ambas miran un álbum de fotos. La interacción sugiere confianza (baja: aprobación), comunicando una relación íntima y cálida entre generaciones.\n"
        f" Caracteristicas:\n"
        f"   - Comunicación no verbal entre personajes\n"
        f"   - Justificación: Especifica CUÁLES personajes interactúan y QUÉ comunican específicamente\n"
        f" 7. Efectos Especiales:\n"
        f" Definición conceptual: Elementos visuales, sonoros o digitales que no se corresponden con una representación mimética de la realidad, sino que introducen una dimensión espectacular, simbólica o fantástica en la escena. Su función semántica puede reforzar emociones, enfatizar acciones o crear transiciones. En la audiodescripción, los efectos especiales deben ser interpretados en términos de su impacto emocional y narrativo, no sólo técnico.\n"
        f" Ejemplo: Una explosión de luz azul recorre el cielo cuando dos personajes se toman de la mano. Este efecto especial subraya una emoción de alegría (alta: éxtasis), acentuando la unión emocional intensa entre los personajes. \n"
        f" Caracteristicas:\n"
        f"   - Elementos visuales artificiales relevantes\n"
        f"   - Justificación: Especifica QUÉ efecto específico y CUÁL es su propósito concreto en la narrativa\n"
        f" 8. Logos y Elementos Gráficos Identificativos:\n"
        f" Definición conceptual: Imágenes, símbolos, logotipos o marcas visuales que identifican una empresa, organización, producto o contenido específico. Su presencia puede aportar información sobre la autoría, el contexto comercial o la intencionalidad comunicativa de la escena. En la audiodescripción, deben ser descritos en cuanto a su forma, colores, ubicación y cualquier elemento distintivo, evitando leer textos pero transmitiendo su función o impacto visual.\n"
        f" Ejemplo: En el centro de la imagen, un logotipo azul y blanco muestra la silueta de un camarógrafo junto a una cámara antigua, enmarcado por un rectángulo celeste. El diseño transmite profesionalismo y está asociado a una productora audiovisual.\n"
        f" Características:\n"
        f"   - Forma y composición visual\n"
        f"   - Colores predominantes\n"
        f"   - Ubicación en la imagen\n"
        f"   - Elementos gráficos distintivos\n"
        f"   - Justificación: Especifica QUÉ logotipo o símbolo se describe y CUÁL es su relevancia o función en la escena\n\n"

        f"Para cada descripción, indica qué mecanismos de atención fueron utilizados, especificando CUÁL o QUÉ elemento específico se está describiendo y proporcionando una justificación detallada de su relevancia en la escena.\n\n"
        f"Pautas obligatorias:\n"
        f"- Usa tiempo presente y voz activa\n"
        f"- NO menciones textos, créditos o elementos escritos\n"
        f"- Describe de lo general a lo específico\n"
        f"- Mantén un tono objetivo y neutral\n"
        f"- Usa oraciones completas y vocabulario claro\n"
        f"- Sé conciso, evitando detalles innecesarios\n"
        f"- Al final de la descripción, lista los mecanismos de atención utilizados"
    )

    # Obtener lista de archivos y ordenarlos por número de frame
    image_files = []
    for filename in os.listdir(image_folder):
        if filename.endswith((".jpg", ".jpeg", ".png")) and (
            filename.startswith("keyframe_")
        ):
            try:
                frame_num = int(filename.split('_')[1].split('.')[0])
                image_files.append((frame_num, filename))
            except:
                continue

    # Ordenar los archivos por número de frame
    image_files.sort(key=lambda x: x[0])

    # Procesar los archivos en orden
    for frame_num, filename in image_files:
        print(f"Procesando archivo: {filename}")
        image_path = os.path.join(image_folder, filename)
        with open(image_path, "rb") as image_file:
            image_data = base64.b64encode(image_file.read()).decode('utf-8')

            response = client.chat.completions.create(
                model="gpt-4o",
                messages=[
                    {"role": "system", "content": prompt},
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": "Describe esta imagen:"},
                            {
                                "type": "image_url",
                                "image_url": {
                                    "url": f"data:image/jpeg;base64,{image_data}"
                                }
                            }
                        ]
                    }
                ],
                max_tokens=1000
            )

        full_response = response.choices[0].message.content
        tokens_used = response.usage.total_tokens

        # Separar la descripción de los mecanismos de atención
        description = ""
        attention_mechanisms = ""

        if "Mecanismos de atención" in full_response:
            parts = full_response.split("Mecanismos de atención")
            description = parts[0].strip()
            attention_mechanisms = "Mecanismos de atención" + parts[1].strip()
        else:
            description = full_response.strip()

        descriptions.append({
            "frame_number": frame_num,
            "filename": filename,
            "description": description,
            "attention_mechanisms": attention_mechanisms,
            "tokens_used": tokens_used
        })

        total_tokens += tokens_used
        print(f"Procesado: {filename} - Tokens usados: {tokens_used}")

    if not descriptions:
        print(f"ADVERTENCIA: No se encontraron imágenes para procesar en {image_folder}")
        print(f"Archivos encontrados: {os.listdir(image_folder)}")

    print(f"Total de tokens usados: {total_tokens}")

    # Guardar el archivo JSON
    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(descriptions, f, ensure_ascii=False, indent=2)

    print("Audiodescripciones guardadas en", output_json)
    return descriptions

# ==========================
# Función: Analizar prosodia
# ==========================
def analyze_prosody(api_key, image_folder, descriptions_json, video_duration, output_json="prosody_analysis.json"):
    client = openai.OpenAI(api_key=api_key)

    with open(descriptions_json, "r", encoding="utf-8") as f:
        descriptions = json.load(f)

    # Usar tanto descripción como mecanismos de atención (como en v7)
    combined_text = " ".join([d["description"] + " " + d["attention_mechanisms"] for d in descriptions])

    # Solicitar a la IA que analice el tono de prosodia adecuado para el video completo
    tone_prompt = (
    f"Analiza al descripción para una audiodescripcion de video para determinar el modelado prosodico \n"
    f"la descripcion del video, mecanismos de atención y modelado prosodico: {combined_text}\n"
    f"Identifica las emociones principales detectadas (deben sumar 100%). Luego, proporciona una única prosodia general para todo el video en el siguiente formato, basado en la tabla de referencia:\n\n"
    f" Identifica prosodia para la imagen basado en la siguiente tabla de referencia:\n\n"
    	"Referencia modelado prosodico:\n\n"
    	"- Emoción principal: Alegría\n"
    	"  - Emoción complementaria: Serenidad\n"
    	"  - Intensidad: Bajo\n"
    	"  - Tono: bajo, estable, sin grandes variaciones\n"
   	"  - Duración: Larga, pausas relajadas\n"
    	"  - Ritmo: Lento, regular, pausado\n\n"
    	"- Emoción principal: Alegría\n"
    	"  - Emoción complementaria: Éxtasis\n"
    	"  - Intensidad: Alta\n"
    	"  - Tono: Alto, con variaciones amplias\n"
    	"  - Duración: Corta, con exclamaciones\n"
    	"  - Ritmo: Rápido, enérgico, dinámico\n\n"
    	"- Emoción principal: Confianza\n"
    	"  - Emoción complementaria: Aceptación\n"
    	"  - Intensidad: Bajo\n"
    	"  - Tono: Medio, cálido, poca variabilidad\n"
    	"  - Duración: Moderada\n"
    	"  - Ritmo: Regular, fluido, sin cambios bruscos\n\n"
    	"- Emoción principal: Confianza\n"
    	"  - Emoción complementaria: Admiración\n"
    	"  - Intensidad: Alto\n"
    	"  - Tono: Alto, a veces ligeramente ascendente\n"
    	"  - Duración: Moderada\n"
    	"  - Ritmo: Enfático, con cierto realce, pero no brusco\n\n"
    	"- Emoción principal: Miedo\n"
    	"  - Emoción complementaria: Aprensión\n"
    	"  - Intensidad: Bajo\n"
    	"  - Tono: Ligero ascenso, posible temblor o quiebre\n"
    	"  - Duración: Moderada, con pausas vacilantes\n"
    	"  - Ritmo: Irregular, inseguro, a veces acelerado\n\n"
    	"- Emoción principal: Miedo\n"
    	"  - Emoción complementaria: Terror\n"
    	"  - Intensidad: Alto\n"
    	"  - Tono: Alto, con quiebres marcados\n"
    	"  - Duración: Corta, entrecortada\n"
    	"  - Ritmo: Muy acelerado o brusco, pausas de jadeo\n\n"
    	"- Emoción principal: Sorpresa\n"
    	"  - Emoción complementaria: Distracción\n"
    	"  - Intensidad: Bajo\n"
    	"  - Tono: Variable, a veces errático\n"
    	"  - Duración: Variable (puede ser corta o larga)\n"
    	"  - Ritmo: Inconsistente, con cambios inesperados\n\n"
    	"- Emoción principal: Sorpresa\n"
    	"  - Emoción complementaria: Asombro\n"
    	"  - Intensidad: Alto\n"
    	"  - Tono: Alto y ascendente al inicio, luego variable\n"
    	"  - Duración: Corta o moderada, con pausas breves\n"
    	"  - Ritmo: Brusco e irregular, marcando sorpresa\n\n"
    	"- Emoción principal: Tristeza\n"
    	"  - Emoción complementaria: Pensamiento profundo\n"
    	"  - Intensidad: Baja y tenue\n"
    	"  - Tono: Medio-bajo, monótono o levemente descendente\n"
    	"  - Duración: Larga, con pausas reflexivas\n"
    	"  - Ritmo: Lento, reflexivo, casi meditativo\n\n"
    	"- Emoción principal: Tristeza\n"
    	"  - Emoción complementaria: Duelo\n"
    	"  - Intensidad: Alto\n"
    	"  - Tono: Bajo, descendente, monótono\n"
    	"  - Duración: Larga, con pausas prolongadas\n"
    	"  - Ritmo: Muy lento, pesado, con silencios marcados\n\n"
    	"- Emoción principal: Disgusto\n"
    	"  - Emoción complementaria: Aburrimiento\n"
    	"  - Intensidad: Bajo\n"
    	"  - Tono: Bajo, con poca variación\n"
    	"  - Duración: larga, habla sostenida\n"
    	"  - Ritmo: Lento, plano, sin cambios de intensidad\n\n"
    	"- Emoción principal: Disgusto\n"
    	"  - Emoción complementaria: Aversión\n"
    	"  - Intensidad: Alto\n"
    	"  - Tono: Medio a alto, seco o cortante\n"
    	"  - Duración: Moderada, a veces con pausas bruscas\n"
    	"  - Ritmo: Cortante, agresivo, enfatizando el rechazo\n\n"
    	"- Emoción principal: Ira\n"
    	"  - Emoción complementaria: Molestia\n"
    	"  - Intensidad: Bajo\n"
    	"  - Tono: Medio, con pequeños picos de tensión\n"
    	"  - Duración: Moderada, frases más cortas\n"
    	"  - Ritmo: Un tanto acelerado, interrumpido, con quiebres\n\n"
    	"- Emoción principal: Ira\n"
    	"  - Emoción complementaria: Furia\n"
    	"  - Intensidad: Alto\n"
    	"  - Tono: Alto, agresivo, con cambios bruscos\n"
    	"  - Duración: Moderada o corta, intercalada con exclamaciones\n"
    	"  - Ritmo: Rápido, explosivo, con picos intensos\n\n"
    	"- Emoción principal: Anticipación\n"
    	"  - Emoción complementaria: Interés\n"
    	"  - Intensidad: Bajo\n"
    	"  - Tono: Alto, ligeramente ascendente\n"
    	"  - Duración: Variable\n"
    	"  - Ritmo: Fluido, ágil, mostrando atención sostenida\n\n"
    	"- Emoción principal: Anticipación\n"
    	"  - Emoción complementaria: Vigilancia\n"
    	"  - Intensidad: Alto\n"
    	"  - Tono: Alto, a veces tenso\n"
    	"  - Duración: Moderada, con pausas para observar\n"
    	"  - Ritmo: Ágil, algo acelerado, atento a cambios\n"
"Formato de salida (por cada emoción principal):\n"
    "Emoción principal: [nombre] [porcentaje]%\n"
    "Emoción complementaria: [nombre]\n"
    "Intensidad: [texto según tabla]\n"
    "Tono: [texto según tabla]\n"
    "Duración: [texto según tabla]\n"
    "Ritmo: [texto según tabla]\n\n"
    "Ejemplo:\n"
    "Emoción principal: Alegría 80%\n"
    "Emoción complementaria: Serenidad\n"
    "Intensidad: Bajoq\n"
    "Tono: Medio-bajo, estable, sin grandes variaciones\n"
    "Duración: Larga, pausas relajadas\n"
    "Ritmo: Lento, regular, pausado\n\n"
    "No de recomendaciones, interpretaciones adicionales ni ajustes sugeridos. Solo responde siguiendo el formato indicado para cada emoción principal detectada.\n\n"
   )


    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": tone_prompt},
           {"role": "user", "content": "Dame una sola prosodia general para todo el video siguiendo el formato especificado"}
        ],
        max_tokens=3000
    )

    prosody_suggestion = response.choices[0].message.content
    tokens_used = response.usage.total_tokens
    print(f"Tokens usados en el análisis de prosodia: {tokens_used}")

    # Intentar extraer los campos principales de la prosodia
    prosody_json = {
        "main_emotion": None,
        "complementary_emotion": None,
        "intensity": None,
        "tone": None,
        "duration": None,
        "rhythm": None,
        "tokens_used": tokens_used
    }
    for line in prosody_suggestion.split('\n'):
        if line.lower().startswith("emoción principal"):
            prosody_json["main_emotion"] = line.split(":")[1].strip()
        elif line.lower().startswith("emoción complementaria"):
            prosody_json["complementary_emotion"] = line.split(":")[1].strip()
        elif line.lower().startswith("intensidad"):
            prosody_json["intensity"] = line.split(":")[1].strip()
        elif line.lower().startswith("tono"):
            prosody_json["tone"] = line.split(":")[1].strip()
        elif line.lower().startswith("duración"):
            prosody_json["duration"] = line.split(":")[1].strip()
        elif line.lower().startswith("ritmo"):
            prosody_json["rhythm"] = line.split(":")[1].strip()
    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(prosody_json, f, ensure_ascii=False, indent=2)
    print("Análisis de prosodia guardado en:", output_json)
    return prosody_json


# ==========================
# Función: Generar resumen de audiodescripciones
# ==========================
def summarize_audiodescriptions(api_key, descriptions_json, video_duration, prosody_analysis_json, output_json="video_summary.json"):
    client = openai.OpenAI(api_key=api_key)
    with open(descriptions_json, "r", encoding="utf-8") as f:
        descriptions = json.load(f)
    # Usar solo las descripciones, no los mecanismos de atención
    combined_text = " ".join([d["description"] for d in descriptions])

    with open(prosody_analysis_json, "r", encoding="utf-8") as f:
        prosody_content = json.load(f)
    selected_emotion = prosody_content.get("complementary_emotion", None)

    emotion_ranges = {
        "Duelo": (7.08, 11.21),
        "Distracción": (7.38, 11.66),
        "Asombro": (8.93, 14.11),
        "Aversión": (8.86, 14.01),
        "Aceptación": (9.44, 14.91),
        "Admiración": (9.08, 14.36),
        "Aprensión": (7.69, 12.15),
        "Vigilancia": (9.44, 14.91),
        "Aburrimiento": (9.08, 14.36),
        "Pensamiento Profundo": (7.61, 12.02),
        "Pensamiento profundo": (7.61, 12.02),  # Agregar variación
        "Molestia": (10.24, 16.18),
        "Serenidad": (9.51, 15.04),
        "Terror": (8.98, 14.18),
        "Interés": (8.41, 13.29),
        "Éxtasis": (10.18, 16.09),
        "Ira": (9.91, 15.65)
    }

    # Calcular rangos de caracteres con valores por defecto si la emoción no se encuentra
    if selected_emotion and selected_emotion in emotion_ranges:
        min_chars_per_sec, max_chars_per_sec = emotion_ranges[selected_emotion]
        min_chars = int(video_duration * min_chars_per_sec)
        max_chars = int(video_duration * max_chars_per_sec)
        print(f"Emoción detectada: {selected_emotion} - Rango: {min_chars} a {max_chars} caracteres")
    else:
        # Valores por defecto si no se encuentra la emoción
        min_chars_per_sec, max_chars_per_sec = 8.0, 12.0  # Valores promedio
        min_chars = int(video_duration * min_chars_per_sec)
        max_chars = int(video_duration * max_chars_per_sec)
        print(f"Emoción no encontrada en rangos: {selected_emotion} - Usando valores por defecto: {min_chars} a {max_chars} caracteres")

    # Asegurar que los rangos sean razonables
    min_chars = max(50, min_chars)  # Mínimo 50 caracteres
    max_chars = max(min_chars + 100, max_chars)  # Al menos 100 caracteres más que el mínimo

    summary_prompt = (
        f"Rol: Eres un Especialista en Audiodescripción para personas ciegas o con baja visión.\n"
        f"Tarea: Genera una descripción general del video basada en las descripciones de los frames individuales.\n\n"
        f"IMPORTANTE - RESTRICCIÓN DE CARACTERES:\n"
        f"La duración del video es {video_duration:.2f} segundos.\n"
        f"La emoción complementaria detectada es: {selected_emotion}\n"
        f"La descripción principal DEBE tener entre {min_chars} y {max_chars} caracteres en total.\n"
        f"NO EXCEDER ESTOS LÍMITES BAJO NINGUNA CIRCUNSTANCIA.\n\n"
        f"Análisis de prosodia del video:\n{json.dumps(prosody_content, ensure_ascii=False, indent=2)}\n\n"
        f"Pautas de estilo:\n"
        f"- Comienza directamente con la descripción, sin introducciones\n"
        f"- Usa lenguaje directo y activo\n"
        f"- Evita frases como 'en la imagen', 'en el video', 'podemos ver', 'se observa', etc.\n"
        f"- Mantén un tono narrativo fluido y natural\n"
        f"- Describe como si estuvieras contando una historia\n"
        f"- Usa verbos en presente y voz activa\n"
        f"- Sé conciso pero descriptivo\n"
        f"- Si en los frames se identifica un logotipo, símbolo o elemento gráfico identificativo, menciónalo y descríbelo también en la descripción general\n\n"
        f"Formato de respuesta:\n"
        f"1. Primero: La descripción principal del video (entre {min_chars} y {max_chars} caracteres)\n"
        f"2. Segundo: Una línea que diga 'MECANISMOS DE ATENCIÓN:' seguida de los mecanismos utilizados\n"
        f"3. Tercero: Una línea que diga 'ESTADÍSTICAS:' seguida de:\n"
        f"   - Emoción complementaria utilizada: {selected_emotion}\n"
        f"   - Número total de caracteres de la descripción principal: [número]\n"
        f"   - Caracteres por segundo: [número]\n"
        f"   - Rango de caracteres permitido para esa emoción: {min_chars} a {max_chars}\n\n"
        f"Descripciones de los frames del video:\n{combined_text}"
    )

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": summary_prompt},
            {"role": "user", "content": "Genera la descripción del video siguiendo exactamente el formato especificado."}
        ],
        max_tokens=1500
    )

    full_summary = response.choices[0].message.content
    tokens_used = response.usage.total_tokens

    # Separar las diferentes secciones del resumen
    main_summary = ""
    attention_mechanisms = ""
    statistics_summary = ""

    # Buscar las diferentes secciones
    if "MECANISMOS DE ATENCIÓN:" in full_summary:
        parts = full_summary.split("MECANISMOS DE ATENCIÓN:")
        main_summary = parts[0].strip()
        remaining = "MECANISMOS DE ATENCIÓN:" + parts[1].strip()

        if "ESTADÍSTICAS:" in remaining:
            stats_parts = remaining.split("ESTADÍSTICAS:")
            attention_mechanisms = stats_parts[0].strip()
            statistics_summary = "ESTADÍSTICAS:" + stats_parts[1].strip()
        else:
            attention_mechanisms = remaining
    else:
        main_summary = full_summary.strip()

    # Calcular estadísticas de caracteres
    if main_summary:
        # Limpiar la descripción principal de cualquier texto adicional
        lines = main_summary.split('\n')
        description_lines = []
        for line in lines:
            line = line.strip()
            if line and not any(keyword in line.upper() for keyword in ["MECANISMOS DE ATENCIÓN:", "ESTADÍSTICAS:", "RESUMEN:"]):
                description_lines.append(line)
            else:
                break

        # Unir las líneas de descripción principal
        main_description_text = ' '.join(description_lines)
        char_count = len(main_description_text)
    else:
        char_count = 0

    chars_per_second = char_count / video_duration if video_duration > 0 else 0

    # Validar el resumen
    if char_count == 0:
        print(f"❌ ERROR: La descripción principal está vacía")
    elif char_count > max_chars:
        print(f"⚠ ADVERTENCIA: El número de caracteres de la descripción ({char_count}) excede el máximo permitido ({max_chars})")
    elif char_count < min_chars:
        print(f"⚠ ADVERTENCIA: El número de caracteres de la descripción ({char_count}) es menor al mínimo permitido ({min_chars})")
    else:
        print(f"✅ Descripción generada correctamente: {char_count} caracteres (rango: {min_chars}-{max_chars})")

    print(f"Tokens usados en el resumen: {tokens_used}")
    print(f"Total de caracteres de la descripción: {char_count}")
    print(f"Caracteres por segundo: {chars_per_second:.2f}")
    print(f"Rango permitido para {selected_emotion}: {min_chars} - {max_chars} caracteres")

    summary_json = {
        "main_summary": main_summary,
        "attention_mechanisms": attention_mechanisms,
        "statistics_summary": statistics_summary,
        "emotion": selected_emotion,
        "char_count": char_count,
        "chars_per_second": chars_per_second,
        "char_range": [min_chars, max_chars],
        "tokens_used": tokens_used
    }

    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(summary_json, f, ensure_ascii=False, indent=2)

    print("Resumen del video guardado en", output_json)
    return summary_json

# ==========================
# Función: Procesar un video individual
# ==========================
def process_video(video_path, output_base_folder=None, api_key=None):
    """Procesa un video: extrae keyframes, genera audiodescripciones, analiza prosodia y resume."""
    if api_key is None:
        print('Se requiere API key para las descripciones y análisis de prosodia')
        return

    if output_base_folder is None:
        output_base_folder = FOLDER_SEGMENTS_DESCRIPTIONS

    video_name = os.path.splitext(os.path.basename(video_path))[0]
    video_output_folder = os.path.join(output_base_folder, video_name)
    os.makedirs(video_output_folder, exist_ok=True)
    print(f'\n{"="*60}')
    print(f'PROCESANDO VIDEO: {video_name}')
    print(f'\n{"="*60}')
    keyframes_folder = os.path.join(video_output_folder, 'keyframes')
    os.makedirs(keyframes_folder, exist_ok=True)
    keyframes, duration = extract_keyframes_clip_adjustable(
        video_path,
        output_folder=keyframes_folder,
        diversity_threshold=0.2,
        similarity_threshold=0.3,
        min_keyframes=3,
        max_keyframes=30,
        frame_sampling_rate=15
    )
    if not keyframes:
        print('No se pudieron extraer keyframes del video')
        return
    print(f'\n{"="*50}')
    print(f'INFORMACIÓN DEL VIDEO:')
    print(f'Ruta: {video_path}')
    print(f'Duración: {duration:.2f} segundos')
    print(f'Formato: {datetime.timedelta(seconds=int(duration))}')
    print(f'\n{"="*50}\n')
    print(f'Se extrajeron {len(keyframes)} keyframes del video')
    audiodesc_json = os.path.join(video_output_folder, 'audiodescriptions.json')
    prosody_json = os.path.join(video_output_folder, 'prosody_analysis.json')
    summary_json = os.path.join(video_output_folder, 'video_summary.json')
    print(f'\nGenerando audiodescripciones para {video_name}...')
    descriptions = generate_audiodescriptions(api_key, keyframes_folder, output_json=audiodesc_json, video_duration=duration)
    print(f'\nAnalizando prosodia para {video_name}...')
    prosody_suggestion = analyze_prosody(api_key, keyframes_folder, audiodesc_json, duration, output_json=prosody_json)
    print(f'\nGenerando resumen para {video_name}...')
    max_retries = 3
    retry_count = 0
    video_summary = None
    while retry_count < max_retries:
        try:
            video_summary = summarize_audiodescriptions(api_key, audiodesc_json, duration, prosody_json, output_json=summary_json)
            if video_summary and 'main_summary' in video_summary:
                main_summary = video_summary['main_summary']
                if is_valid_summary(main_summary):
                    print(f'✓ Resumen generado exitosamente y validado')
                    break
                else:
                    print(f'⚠ Resumen inválido detectado: {main_summary[:50]}...')
                    print(f'Reintentando generación del resumen (intento {retry_count + 2}/{max_retries})...')
            else:
                print(f'⚠ Resumen vacío o sin main_summary, reintentando...')
        except Exception as e:
            print(f'Error generando resumen: {str(e)}')
            print(f'Reintentando... (intento {retry_count + 2}/{max_retries})')
        retry_count += 1
        if retry_count < max_retries:
            print('Esperando 5 segundos antes del reintento...')
            time.sleep(5)
    if retry_count >= max_retries:
        print(f'❌ No se pudo generar un resumen válido después de {max_retries} intentos')
        if video_summary and 'main_summary' in video_summary:
            print(f'Último resumen generado: {video_summary["main_summary"][:100]}...')
    else:
        print(f'✅ Resumen final válido generado en {retry_count + 1} intentos')
    print(f'\n{"="*50}')
    print(f'PROCESO COMPLETADO PARA: {video_name}')
    print(f'\n{"="*50}')
    print(f'Se han generado en {video_output_folder}:')
    print(f'1. Keyframes en {keyframes_folder}')
    print(f'2. Descripciones en {audiodesc_json}')
    print(f'3. Resumen del video en {summary_json}')
    print(f'4. Análisis de prosodia en {prosody_json}')
    print(f'\n{"="*50}\n')
    return video_output_folder

# ==========================
# Función: Procesar todos los videos de la carpeta input (Drive)
# ==========================
def process_all_videos_drive(videos_folder='videos', output_base_folder=None, api_key=None):
    """Procesa todos los videos en la carpeta de Google Drive y guarda los resultados en FOLDER_SEGMENTS_DESCRIPTIONS."""
    if not os.path.exists(videos_folder):
        print(f'Error: La carpeta {videos_folder} no existe')
        return

    if output_base_folder is None:
        output_base_folder = FOLDER_SEGMENTS_DESCRIPTIONS

    video_extensions = ['.mp4', '.avi', '.mov', '.mkv', '.wmv', '.flv']
    video_files = []
    for filename in os.listdir(videos_folder):
        if any(filename.lower().endswith(ext) for ext in video_extensions):
            # Solo procesar videos que contengan "segmento" en el nombre
            if "segmento" in filename.lower():
                video_path = os.path.join(videos_folder, filename)
                video_files.append(video_path)
    if not video_files:
        print(f'No se encontraron archivos de video que contengan "segmento" en {videos_folder}')
        return
    video_files.sort()
    print(f'Se encontraron {len(video_files)} videos que contienen "segmento" para procesar:')
    for video_path in video_files:
        print(f'  - {os.path.basename(video_path)}')
    print(f'\n{"="*60}')
    print('INICIANDO PROCESAMIENTO DE TODOS LOS VIDEOS')
    print(f'\n{"="*60}')
    processed_videos = []
    failed_videos = []
    for i, video_path in enumerate(video_files, 1):
        try:
            print(f'\nProcesando video {i}/{len(video_files)}: {os.path.basename(video_path)}')
            output_folder = process_video(video_path, output_base_folder, api_key)
            if output_folder:
                processed_videos.append((video_path, output_folder))
        except Exception as e:
            print(f'Error procesando {os.path.basename(video_path)}: {str(e)}')
            failed_videos.append(video_path)
    print(f'\n{"="*60}')
    print('RESUMEN FINAL DEL PROCESAMIENTO')
    print(f'\n{"="*60}')
    print(f'Videos procesados exitosamente: {len(processed_videos)}')
    print(f'Videos con errores: {len(failed_videos)}')
    if processed_videos:
        print(f'\nVideos procesados exitosamente:')
        for video_path, output_folder in processed_videos:
            print(f'  ✓ {os.path.basename(video_path)} → {output_folder}')
    if failed_videos:
        print(f'\nVideos con errores:')
        for video_path in failed_videos:
            print(f'  ✗ {os.path.basename(video_path)}')
    print(f'\nTodos los archivos de salida se encuentran en la carpeta: {output_base_folder}')
    print(f'\n{"="*60}')

In [ ]:
# @title
# ==========================
# Conexión y configuración de Google Drive
# ==========================
from googleapiclient.discovery import build
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
import io
from googleapiclient.http import MediaIoBaseDownload
import traceback # Importar para obtener información detallada del error

def connect_to_drive():
    """Establece conexión con Google Drive usando las credenciales configuradas."""
    creds = Credentials(
        token=None,
        refresh_token=REFRESH_TOKEN,
        token_uri="https://oauth2.googleapis.com/token",
        client_id=CLIENT_ID,
        client_secret=CLIENT_SECRET
    )

    if creds.expired:
        try:
            creds.refresh(Request())
        except Exception as e:
            print(f"❌ Error al refrescar las credenciales de Drive: {str(e)}")
            traceback.print_exc()
            raise

    service = build('drive', 'v3', credentials=creds)
    print("✅ Conexión exitosa con Google Drive")
    return service

def download_videos_from_drive(service, folder_id, local_folder='videos'):
    """
    Descarga videos que contengan 'segmento' de la carpeta de Google Drive especificada,
    buscando en la carpeta principal y sus subcarpetas directas.
    """
    os.makedirs(local_folder, exist_ok=True)

    print(f"🔎 Buscando videos que contengan 'segmento' en la carpeta de Drive ID: {folder_id} y subcarpetas directas")

    items = []
    search_folder_ids = [folder_id] # Empezar buscando en la carpeta principal

    try:
        # Obtener IDs de subcarpetas directas
        subfolder_query = f"parents in '{folder_id}' and mimeType='application/vnd.google-apps.folder' and trashed = false"
        subfolder_results = service.files().list(q=subfolder_query, fields="files(id, name)").execute()
        subfolder_items = subfolder_results.get('files', [])

        print(f"📁 Encontradas {len(subfolder_items)} subcarpetas directas en folder ID: {folder_id}")
        for subfolder in subfolder_items:
            print(f"  - {subfolder.get('name')} (ID: {subfolder.get('id')})")
            search_folder_ids.append(subfolder.get('id'))


        # Buscar archivos de video en todas las carpetas (principal + subcarpetas directas)
        for current_folder_id in search_folder_ids:
            print(f"\n🔎 Buscando archivos de video en folder ID: {current_folder_id}")
            query = f"parents in '{current_folder_id}' and (mimeType = 'video/mp4' or mimeType = 'video/x-matroska' or mimeType contains 'video/' or mimeType = 'application/octet-stream') and trashed = false"

            print(f"🔑 Query used for folder {current_folder_id}: {query}") # Log the query being used

            current_results = service.files().list(
                q=query,
                pageSize=1000,
                fields="nextPageToken, files(id, name, mimeType, size)"
            ).execute()
            items.extend(current_results.get('files', []))
            print(f"📄 Encontrados {len(current_results.get('files', []))} archivos con tipo MIME de video en folder ID: {current_folder_id}")


        print(f"\n📄 Total de archivos con tipo MIME de video encontrados en las carpetas buscadas: {len(items)}")
        print("Lista completa de archivos encontrados por la API:")
        for item in items:
             print(f"  - {item.get('name')} ({item.get('id')}) - MIME: {item.get('mimeType')}")


        if not items:
            print(f"❌ No se encontraron videos con tipo MIME de video en las carpetas buscadas.")
            return []

        # Filtrar solo videos que contengan "segmento" (insensible a mayúsculas/minúsculas)
        filter_term = "segmento"
        segment_videos = [item for item in items if filter_term.lower() in item.get('name', '').lower()]

        print(f"\n🔍 Filtrando por el término '{filter_term}': Encontrados {len(segment_videos)} videos.")
        if segment_videos:
            print("📝 Videos seleccionados para descargar:")
            for item in segment_videos:
                 print(f"  - {item.get('name')} ({item.get('id')})")


        if not segment_videos:
            print(f"❌ No se encontraron videos que contengan '{filter_term}' en las carpetas de Drive buscadas.")
            return []

        print(f"📥 Descargando {len(segment_videos)} videos con '{filter_term}' de Google Drive...")
        downloaded_files = []

        for item in segment_videos:
            file_name = item['name']
            file_id = item['id']
            file_size = item.get('size')
            size_str = f"({file_size} bytes)" if file_size else ""


            local_path = os.path.join(local_folder, file_name)

            # Verificar si el archivo ya existe localmente y tiene el mismo tamaño
            if os.path.exists(local_path):
                local_size = os.path.getsize(local_path)
                # Convert file_size to int safely if it exists
                drive_file_size_int = int(file_size) if file_size is not None else -1
                if file_size is not None and drive_file_size_int == local_size:
                    print(f"⏭️  Archivo ya existe localmente con el mismo tamaño: {file_name}")
                    downloaded_files.append(local_path)
                    continue
                else:
                    print(f"🔄 Archivo existe localmente pero tamaño diferente o desconocido, descargando de nuevo: {file_name}")


            try:
                print(f"📥 Descargando: {file_name} {size_str}")

                # Descargar archivo
                request = service.files().get_media(fileId=file_id)
                file_io = io.BytesIO()
                downloader = MediaIoBaseDownload(file_io, request)

                done = False
                while done is False:
                    status, done = downloader.next_chunk()
                    if status:
                        progress = int(status.progress() * 100)
                        print(f"   📊 Progreso: {progress}%", end='\r')

                # Guardar archivo
                with open(local_path, 'wb') as f:
                    f.write(file_io.getvalue())

                downloaded_files.append(local_path)
                print(f"\n✅ Descargado: {file_name}") # Nueva línea después del progreso

            except Exception as e:
                print(f"❌ Error descargando {file_name}: {str(e)}")
                traceback.print_exc() # Imprimir traceback detallado

        print(f"\n📦 Total videos descargados: {len(downloaded_files)}")
        return downloaded_files

    except Exception as e:
        print(f"❌ Error al listar o descargar archivos de Drive: {str(e)}")
        traceback.print_exc() # Imprimir traceback si falla la lista inicial
        return []


def upload_results_to_drive(service, local_folder, folder_id):
    """Sube los resultados del procesamiento a la carpeta 'segments descriptions' en Google Drive."""
    print(f"\n📤 Subiendo resultados desde '{local_folder}' a Google Drive folder ID: {folder_id}")

    if not os.path.exists(local_folder):
        print(f"❌ La carpeta local de resultados '{local_folder}' no existe.")
        return

    # Obtener lista de subcarpetas (una por cada video procesado)
    processed_video_folders = [d for d in os.listdir(local_folder) if os.path.isdir(os.path.join(local_folder, d))]

    if not processed_video_folders:
        print(f"⚠️ No se encontraron subcarpetas de resultados en '{local_folder}' para subir.")
        return


    for video_folder in processed_video_folders:
        video_folder_path = os.path.join(local_folder, video_folder)
        print(f"\n📤 Subiendo resultados de: {video_folder}")

        try:
            # Crear carpeta en Drive si no existe
            drive_folder_id = create_drive_folder(service, video_folder, folder_id)
            if not drive_folder_id:
                 print(f"❌ No se pudo obtener/crear la carpeta en Drive para {video_folder}. Saltando subida para este video.")
                 continue

            # Subir archivos de la carpeta
            for item_name in os.listdir(video_folder_path):
                item_path = os.path.join(video_folder_path, item_name)

                if os.path.isfile(item_path):
                    upload_file_to_drive(service, item_path, item_name, drive_folder_id)
                elif os.path.isdir(item_path):
                    # Subir carpeta de keyframes
                    print(f"📂 Subiendo contenido de subcarpeta: {item_name}")
                    keyframes_folder_id = create_drive_folder(service, item_name, drive_folder_id)
                    if keyframes_folder_id:
                         for keyframe_file in os.listdir(item_path):
                             keyframe_path = os.path.join(item_path, keyframe_file)
                             if os.path.isfile(keyframe_path):
                                 upload_file_to_drive(service, keyframe_path, keyframe_file, keyframes_folder_id)
                    else:
                         print(f"❌ No se pudo obtener/crear la carpeta en Drive para keyframes '{item_name}'. Saltando subida de keyframes.")

        except Exception as e:
            print(f"❌ Error general al subir resultados de {video_folder}: {str(e)}")
            traceback.print_exc()


def create_drive_folder(service, folder_name, parent_id):
    """Crea una carpeta en Google Drive."""
    # Verificar si ya existe una carpeta con ese nombre
    existing_query = f"name='{folder_name}' and parents in '{parent_id}' and mimeType='application/vnd.google-apps.folder' and trashed = false"
    existing_folders = service.files().list(q=existing_query, fields="files(id, name)").execute()

    if existing_folders.get('files'):
        # Ya existe, devolver el ID de la existente
        folder_id = existing_folders['files'][0]['id']
        # print(f"📁 Carpeta existente encontrada: '{folder_name}' (ID: {folder_id})")
        return folder_id

    # No existe, crear nueva carpeta
    folder_metadata = {
        'name': folder_name,
        'mimeType': 'application/vnd.google-apps.folder',
        'parents': [parent_id]
    }

    try:
        folder = service.files().create(body=folder_metadata, fields='id').execute()
        folder_id = folder.get('id')
        print(f"📁 Carpeta creada en Drive: '{folder_name}' (ID: {folder_id})")
        return folder_id
    except Exception as e:
        print(f"❌ Error creando carpeta '{folder_name}' en Drive: {str(e)}")
        traceback.print_exc()
        return None

def upload_file_to_drive(service, file_path, file_name, folder_id):
    """Sube un archivo a Google Drive."""
    from googleapiclient.http import MediaFileUpload

    if not os.path.exists(file_path):
        print(f"⚠️ Archivo local no encontrado, no se puede subir: {file_path}")
        return None

    try:
        # Verificar si el archivo ya existe
        existing_query = f"name='{file_name}' and parents in '{folder_id}' and trashed = false"
        existing_files = service.files().list(q=existing_query, fields="files(id, name)").execute()

        if existing_files.get('files'):
            # Si existe, podemos optar por actualizarlo o saltarlo.
            # Por ahora, vamos a saltar si ya existe.
            file_id = existing_files['files'][0]['id']
            # print(f"⏭️  Archivo ya existe en Drive: '{file_name}' (ID: {file_id}). Saltando subida.")
            return file_id

        # No existe, subir nuevo archivo
        file_metadata = {
            'name': file_name,
            'parents': [folder_id]
        }

        media = MediaFileUpload(file_path, resumable=True)
        file = service.files().create(body=file_metadata, media_body=media, fields='id, name').execute()
        print(f"✅ Subido: '{file.get('name')}' (ID: {file.get('id')})")
        return file.get('id')
    except Exception as e:
        print(f"❌ Error subiendo '{file_name}' a Drive: {str(e)}")
        traceback.print_exc()
        return None
print("🔧 Funciones de Conexión y configuración de Google Drive definidas")

In [ ]:
# @title
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title
# ==========================
# Configuración inicial y flujo principal
# ==========================

# Configurar carpetas y mostrar IDs
print("🚀 CONFIGURACIÓN INICIAL DE CARPETAS")
print("="*50)

try:
    folder_input_id, folder_segments_descriptions_id = setup_folders()
    print(f"✅ Configuración completada:")
    print(f"   📁 Folder Input ID: {folder_input_id}")
    print(f"   📁 Folder Segments Descriptions ID: {folder_segments_descriptions_id}")
except Exception as e:
    print(f"❌ Error en la configuración: {str(e)}")
    print("Por favor, verifica la conexión a Drive y vuelve a intentar.")

In [ ]:
# @title
# ==========================
# FLUJO COMPLETO CON GOOGLE DRIVE
# ==========================

def main_processing_workflow(api_key, use_drive=True, local_videos_folder='videos'):
    """
    Flujo principal de procesamiento con opciones de Google Drive.

    Args:
        api_key: API key de OpenAI
        use_drive: Si usar Google Drive (True) o carpeta local (False)
        local_videos_folder: Carpeta local de videos si use_drive=False
    """

    print(f"\n🚀 INICIANDO FLUJO DE PROCESAMIENTO DE AUDIODESCRIPCIONES")
    print(f'{"="*70}')

    # Verificar que las carpetas estén configuradas
    if FOLDER_INPUT is None or FOLDER_SEGMENTS_DESCRIPTIONS is None:
        print("❌ Error: Las carpetas no han sido configuradas. Ejecuta setup_folders() primero.")
        return

    # Paso 1: Configurar origen de videos
    if use_drive:
        print("📡 Modo: Google Drive")
        print(f"📁 Usando Folder Input: {FOLDER_INPUT}")
        print(f"📁 Guardando en Folder Segments Descriptions: {FOLDER_SEGMENTS_DESCRIPTIONS}")

        try:
            service = connect_to_drive()
            video_files = download_videos_from_drive(service, FOLDER_INPUT, local_videos_folder)

            if not video_files:
                print("❌ No se pudieron descargar videos de Drive")
                return

        except Exception as e:
            print(f"❌ Error conectando con Google Drive: {str(e)}")
            print("🔄 Cambiando a modo local...")
            use_drive = False

    if not use_drive:
        print(f"💻 Modo: Carpeta local ({local_videos_folder})")
        if not os.path.exists(local_videos_folder):
            print(f"❌ La carpeta {local_videos_folder} no existe")
            return

    # Paso 2: Procesar videos
    print(f"\n🎬 INICIANDO PROCESAMIENTO DE VIDEOS")
    print(f'{"="*50}')

    # Configurar carpeta de salida local temporal
    temp_output_folder = 'temp_segments_descriptions'

    try:
        # Usar la función existente para procesar videos
        process_all_videos_drive(
            videos_folder=local_videos_folder,
            output_base_folder=temp_output_folder,
            api_key=api_key
        )

        # Paso 3: Subir resultados a Drive (si está habilitado)
        if use_drive:
            print(f"\n📤 SUBIENDO RESULTADOS A GOOGLE DRIVE")
            print(f'{"="*50}')
            try:
                upload_results_to_drive(service, temp_output_folder, FOLDER_SEGMENTS_DESCRIPTIONS)
                print("✅ Resultados subidos exitosamente a Google Drive")
                print(f"📁 ID de la carpeta 'segments descriptions': {FOLDER_SEGMENTS_DESCRIPTIONS}")

                # Limpiar archivos temporales locales (opcional)
                import shutil
                if os.path.exists(temp_output_folder):
                    shutil.rmtree(temp_output_folder)
                    print("🧹 Archivos temporales locales eliminados")

            except Exception as e:
                print(f"❌ Error subiendo resultados a Drive: {str(e)}")
                print(f"📁 Los resultados están disponibles localmente en: {temp_output_folder}")
        else:
            print(f"📁 Resultados guardados localmente en: {temp_output_folder}")

        print(f"\n🎉 PROCESAMIENTO COMPLETADO EXITOSAMENTE")
        print(f'{"="*70}')
        if use_drive:
            print(f"📁 Carpeta de resultados en Drive: {FOLDER_SEGMENTS_DESCRIPTIONS}")

    except Exception as e:
        print(f"❌ Error durante el procesamiento: {str(e)}")
        return

# Configurar y ejecutar el flujo principal
if __name__ == '__main__':
    # Verificar que las carpetas estén configuradas
    if FOLDER_INPUT is None or FOLDER_SEGMENTS_DESCRIPTIONS is None:
        print("⚠️  ATENCIÓN: Antes de ejecutar el procesamiento, debes ejecutar la celda anterior para configurar las carpetas.")
        print("   Ejecuta la celda que contiene setup_folders() para solicitar el ID del folder input.")
    else:
        # Configuración
        API_KEY = os.environ.get("API_KEY", "")  # credencial externa

        # Ejecutar flujo
        main_processing_workflow(
            api_key=API_KEY,
            use_drive=True,  # Cambiar a False para usar solo carpetas locales
            local_videos_folder='videos'
        )

# Organizar datos para narracion



In [ ]:
# @title
# ==========================
# Crear carpeta 'Input' y mover carpetas existentes
# ==========================

def organize_drive_folders_within_parent(service, parent_folder_to_organize_id, input_folder_name='Input', exclude_folder_names=None):
    """
    Crea una carpeta 'Input' dentro de una carpeta padre especificada
    y mueve todas las demás carpetas existentes *dentro de esa carpeta padre*
    (excepto las excluidas) dentro de la carpeta 'Input'.

    Args:
        service: Objeto del servicio de Google Drive.
        parent_folder_to_organize_id: ID de la carpeta padre donde se creará 'Input'
                                      y cuyas subcarpetas se organizarán.
        input_folder_name: Nombre de la carpeta que se creará para contener las otras carpetas.
        exclude_folder_names: Lista de nombres de carpetas a excluir de la organización.
                               Por defecto, excluye solo la carpeta 'Input' que se va a crear.

    Returns:
        ID de la carpeta 'Input' creada o encontrada dentro del padre especificado.
    """
    if exclude_folder_names is None:
        # Excluir solo la carpeta Input que se va a crear DENTRO del padre
        exclude_folder_names = [input_folder_name]

    print(f"\n📂 Organizando carpetas en Drive DENTRO del folder padre ID: {parent_folder_to_organize_id}")

    try:
        # 1. Crear o encontrar la carpeta 'Input' dentro del padre especificado
        input_folder_id = create_drive_folder(service, input_folder_name, parent_folder_to_organize_id)
        if not input_folder_id:
            print(f"❌ No se pudo crear o encontrar la carpeta '{input_folder_name}' dentro de {parent_folder_to_organize_id}.")
            return None

        # 2. Listar todas las carpetas *dentro* del folder padre especificado
        query = f"parents in '{parent_folder_to_organize_id}' and mimeType='application/vnd.google-apps.folder' and trashed = false"
        existing_folders = service.files().list(q=query, fields="files(id, name)").execute()
        folders_to_organize = existing_folders.get('files', [])

        if not folders_to_organize:
            print(f"⚠️ No se encontraron otras carpetas dentro del nivel padre ({parent_folder_to_organize_id}) para organizar.")
            return input_folder_id

        print(f"🔎 Encontradas {len(folders_to_organize)} carpetas dentro del nivel padre.")
        print("📝 Carpetas encontradas:")
        for folder in folders_to_organize:
            print(f"  - {folder.get('name')} (ID: {folder.get('id')})")

        # 3. Mover carpetas (excepto las excluidas y la propia carpeta Input) a la carpeta 'Input'
        moved_count = 0
        for folder in folders_to_organize:
            folder_name = folder.get('name')
            folder_id = folder.get('id')

            # Excluir la carpeta 'Input' que acabamos de crear y cualquier otra especificada
            if folder_id != input_folder_id and folder_name not in exclude_folder_names:
                try:
                    # Mover la carpeta a la carpeta 'Input'
                    # Primero, removemos el padre actual (el parent_folder_to_organize_id)
                    service.files().update(fileId=folder_id, removeParents=parent_folder_to_organize_id, fields='id, parents').execute()

                    # Luego, agregamos el nuevo padre (la carpeta 'Input')
                    service.files().update(fileId=folder_id, addParents=input_folder_id, fields='id, parents').execute()

                    print(f"➡️  Movida carpeta '{folder_name}' (ID: {folder_id}) a '{input_folder_name}'.")
                    moved_count += 1
                except Exception as e:
                    print(f"❌ Error moviendo carpeta '{folder_name}' (ID: {folder_id}): {str(e)}")
                    #traceback.print_exc() # Opcional: imprimir traceback detallado

        print(f"\n✅ Organización completada. Se movieron {moved_count} carpeta(s) a '{input_folder_name}'.")
        return input_folder_id

    except Exception as e:
        print(f"❌ Error general durante la organización de carpetas: {str(e)}")
        #traceback.print_exc() # Opcional: imprimir traceback detallado
        return None

# --- Ejecutar la función de organización ---
# La carpeta padre donde queremos organizar es 'segments descriptions'
try:
    print("\n⚙️  Iniciando organización de carpetas dentro de 'segments descriptions'...")
    service = connect_to_drive() # Asegurarse de tener una conexión activa

    if FOLDER_SEGMENTS_DESCRIPTIONS:
        print(f"✅ Usando 'segments descriptions' como carpeta padre para organizar: ID {FOLDER_SEGMENTS_DESCRIPTIONS}")
        # Ejecutar la función para organizar las carpetas DENTRO de FOLDER_SEGMENTS_DESCRIPTIONS
        # Excluimos 'segments descriptions' de la lista de carpetas a mover (aunque no debería estar dentro de sí misma)
        input_folder_id = organize_drive_folders_within_parent(
            service,
            FOLDER_SEGMENTS_DESCRIPTIONS,
            input_folder_name='Input',
            exclude_folder_names=['Input'] # Solo excluimos la carpeta 'Input' que estamos creando allí
        )

        if input_folder_id:
            print(f"\n🎉 La carpeta 'Input' ha sido creada dentro de 'segments descriptions' (ID: {input_folder_id}) y las carpetas han sido organizadas.")
            # Opcional: Si el flujo posterior necesita el ID de esta nueva carpeta Input, puedes asignarlo.
            # FOLDER_INPUT_ORGANIZED = input_folder_id # Usar un nuevo nombre de variable para evitar confusión
        else:
             print("\n❌ No se pudo completar la organización de carpetas dentro de 'segments descriptions'.")

    else:
        print("❌ El ID de la carpeta 'segments descriptions' (FOLDER_SEGMENTS_DESCRIPTIONS) no está configurado.")
        print("Asegúrate de ejecutar la celda de configuración inicial primero.")

except Exception as e:
    print(f"❌ Error general al conectar con Drive o durante la organización: {str(e)}")

In [ ]:
# @title
# ==========================
# Copiar video original y archivo de diarización a la carpeta Input
# ==========================

def copy_files_to_input_folder(service, video_source_folder_id, diarization_source_folder_id, target_input_folder_id):
    """
    Copia un video (el primero encontrado) de una carpeta de origen y un archivo
    específico ('video_silent_segments_with_scenes.json') de otra carpeta de origen
    a la carpeta Input de destino.

    Args:
        service: Objeto del servicio de Google Drive.
        video_source_folder_id: ID de la carpeta de origen del video.
        diarization_source_folder_id: ID de la carpeta de origen del archivo de diarización.
        target_input_folder_id: ID de la carpeta Input de destino.
    """
    print("\n📁 Copiando video y archivo de diarización a la carpeta Input...")

    # --- Copiar el video ---
    print("\n🔎 Buscando video en la carpeta de origen...")
    video_query = f"parents in '{video_source_folder_id}' and (mimeType contains 'video/' or mimeType = 'application/octet-stream') and trashed = false"
    video_results = service.files().list(q=video_query, fields="files(id, name, mimeType)").execute()
    videos = video_results.get('files', [])

    if not videos:
        print(f"❌ No se encontró ningún archivo de video en la carpeta ID: {video_source_folder_id}")
    else:
        # Tomar el primer video encontrado
        video_item = videos[0]
        video_file_id = video_item['id']
        video_name = video_item['name']
        print(f"✅ Video encontrado: '{video_name}' (ID: {video_file_id})")

        try:
            # Copiar el video a la carpeta Input
            copied_video_metadata = {
                'name': video_name, # Mantener el nombre original del video
                'parents': [target_input_folder_id]
            }
            copied_video = service.files().copy(fileId=video_file_id, body=copied_video_metadata, fields='id, name, parents').execute()
            print(f"➡️  Video '{video_name}' copiado a la carpeta Input. Nuevo ID: {copied_video.get('id')}")
        except Exception as e:
            print(f"❌ Error copiando video '{video_name}': {str(e)}")
            #traceback.print_exc() # Opcional: imprimir traceback detallado


    # --- Copiar el archivo de diarización ---
    print("\n🔎 Buscando archivo de diarización 'video_silent_segments_with_scenes.json'...")
    video_name_no_ext = video_name.split('.')[0] # Eliminar la extensión del video
    diarization_filename = 'silent_segments_with_scenes.json'
    #add video_name_no_ext to diarization_filename
    diarization_filename = f"{video_name_no_ext}_{diarization_filename}"
    diarization_query = f"parents in '{diarization_source_folder_id}' and name='{diarization_filename}' and trashed = false"
    diarization_results = service.files().list(q=diarization_query, fields="files(id, name)").execute()
    diarization_files = diarization_results.get('files', [])

    if not diarization_files:
        print(f"❌ No se encontró el archivo '{diarization_filename}' en la carpeta ID: {diarization_source_folder_id}")
    else:
        # Tomar el primer archivo encontrado (debería ser único por nombre en la carpeta)
        diarization_item = diarization_files[0]
        diarization_file_id = diarization_item['id']
        print(f"✅ Archivo de diarización encontrado: '{diarization_filename}' (ID: {diarization_file_id})")

        try:
            # Copiar el archivo a la carpeta Input y renombrarlo a 'Main_video.json'
            copied_diarization_metadata = {
                'name': 'Main_video.json', # Renombrar el archivo copiado
                'parents': [target_input_folder_id]
            }
            copied_diarization = service.files().copy(fileId=diarization_file_id, body=copied_diarization_metadata, fields='id, name, parents').execute()
            print(f"➡️  Archivo '{diarization_filename}' copiado a la carpeta Input como 'Main_video.json'. Nuevo ID: {copied_diarization.get('id')}")
        except Exception as e:
            print(f"❌ Error copiando archivo de diarización '{diarization_filename}': {str(e)}")
            #traceback.print_exc() # Opcional: imprimir traceback detallado

    print("\n✅ Proceso de copia completado.")


# --- Ejecutar el proceso de copia ---
try:
    print("\n⚙️  Iniciando proceso de copia de archivos...")
    service = connect_to_drive() # Asegurarse de tener una conexión activa

    # Validar que FOLDER_SEGMENTS_DESCRIPTIONS esté configurado
    if not FOLDER_SEGMENTS_DESCRIPTIONS:
        print("❌ El ID de la carpeta 'segments descriptions' (FOLDER_SEGMENTS_DESCRIPTIONS) no está configurado.")
        print("Asegúrate de ejecutar la celda de configuración inicial primero.")
    else:
        # Primero, necesitamos encontrar el ID de la carpeta 'Input' dentro de FOLDER_SEGMENTS_DESCRIPTIONS
        print(f"🔎 Buscando la carpeta 'Input' dentro de 'segments descriptions' (ID: {FOLDER_SEGMENTS_DESCRIPTIONS})...")
        input_folder_query = f"name='Input' and parents in '{FOLDER_SEGMENTS_DESCRIPTIONS}' and mimeType='application/vnd.google-apps.folder' and trashed = false"
        input_folder_results = service.files().list(q=input_folder_query, fields="files(id)").execute()
        input_folders = input_folder_results.get('files', [])

        if not input_folders:
            print(f"❌ No se encontró la carpeta 'Input' dentro de 'segments descriptions' (ID: {FOLDER_SEGMENTS_DESCRIPTIONS}).")
            print("Asegúrate de que la organización de carpetas se haya ejecutado correctamente.")
        else:
            input_folder_id = input_folders[0]['id']
            print(f"✅ Carpeta 'Input' encontrada. ID: {input_folder_id}")

            # Pedir al usuario los IDs de las carpetas de origen
            video_source_folder_id = input("\n📁 Ingresa el ID de la carpeta de Google Drive donde se encuentra el video original: ").strip()
            diarization_source_folder_id = input("📁 Ingresa el ID de la carpeta de Google Drive donde se encuentra el archivo de resultados de diarización (video_silent_segments_with_scenes.json): ").strip()

            if not video_source_folder_id or not diarization_source_folder_id:
                print("❌ Debes ingresar ambos IDs de carpeta de origen.")
            else:
                # Ejecutar la función de copia
                copy_files_to_input_folder(
                    service,
                    video_source_folder_id,
                    diarization_source_folder_id,
                    input_folder_id # Usamos el ID de la carpeta 'Input' encontrada
                )

except Exception as e:
    print(f"❌ Error general durante el proceso de copia: {str(e)}")
    #traceback.print_exc() # Opcional: imprimir traceback detallado

# **Pipeline de Generación y Post-Producción de Audio**

Este notebook integra un flujo de trabajo completo para la creación de pistas de audio descriptivo (AD). El proceso automatizado incluye:

1.  **Configuración del Entorno**: Instalación de todas las librerías necesarias, incluyendo Coqui TTS, OpenAI Whisper, Moviepy y librerías de análisis de audio.
2.  **Carga de Tareas**: Lectura de archivos `JSON` para extraer los textos a sintetizar y la emoción principal deseada para cada segmento de video.
3.  **Generación de Audio (TTS)**: Uso del modelo `XTTS v2` de Coqui para generar las voces a partir del texto, utilizando audios de referencia para clonar la emoción.
4.  **Verificación y Calidad**: Empleo de la API de Whisper para transcribir el audio generado y asegurar que coincide con el texto original mediante una comparación difusa.
5.  **Análisis Prosódico (Opcional)**: Evaluación objetiva de las características del audio generado (ritmo, intensidad, entonación) utilizando `openSMILE` para generar reportes y gráficas.
6.  **Mezcla Profesional Final**: Un script de post-producción que:
    * Extrae el audio original del video.
    * Aplica un efecto "ducking" para atenuar el audio base durante los diálogos.
    * Sincroniza e inserta los audios generados en los momentos exactos definidos en un archivo `JSON` de tiempos.
    * Exporta tanto la pista de audio final como un nuevo video con el audio ya mezclado.

---



## 1. Configuración del Entorno de Ejecución

Esta primera celda de código instala **todas** las librerías necesarias para el pipeline completo. Es fundamental ejecutarla al inicio de cada sesión.

**Importante**: Este notebook está diseñado para funcionar con **Python 3.11; Versión del entorno de ejecución 2025.07**.

In [ ]:
# @title ⚙️ Instalación de Dependencias Base 1/2
import sys
import time
import threading
import subprocess

def instalar_paquete_silencioso(comando, descripcion="Instalando dependencias"):
    """
    Ejecuta un comando de terminal en segundo plano suprimiendo la salida estándar
    y muestra una animación de transición de emojis en la celda de Colab.
    """
    # Variables de control para el hilo
    proceso_terminado = False
    emojis_animacion = ["⏳", "⌛"] # Emojis de transición
    emoji_exito = "✅"
    emoji_error = "❌"

    # Función que se ejecuta en el hilo paralelo
    def mostrar_progreso():
        i = 0
        while not proceso_terminado:
            # El retorno de carro '\r' sobrescribe la misma línea
            sys.stdout.write(f"\r{emojis_animacion[i % len(emojis_animacion)]} {descripcion}...")
            sys.stdout.flush()
            time.sleep(0.4)
            i += 1

    # Iniciar la animación en un hilo separado
    hilo_animacion = threading.Thread(target=mostrar_progreso)
    hilo_animacion.start()

    # Ejecutar el comando de instalación bloqueando la salida de consola
    resultado = subprocess.run(
        comando,
        shell=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

    # Detener el hilo de animación
    proceso_terminado = True
    hilo_animacion.join()

    # Evaluar el código de salida del subproceso y mostrar el estado final
    if resultado.returncode == 0:
        sys.stdout.write(f"\r{emoji_exito} {descripcion}... ¡✅ Completado exitosamente! \n")
    else:
        sys.stdout.write(f"\r{emoji_error} {descripcion}... ¡❌ Fallo en la ejecución!         \n")


instalar_paquete_silencioso(
    comando="pip install accelerate==0.28.0 pandas==1.5.3 numpy==1.26.4 --force-reinstall llvmlite==0.41.1 numba==0.58.1 -q",
    descripcion="Instalando Pandas y Numpy"
)

instalar_paquete_silencioso(
   # transformers==4.38.2 --force-reinstall
    comando="pip install TTS -q",
    descripcion="Instalando TTS"
)


In [ ]:
# @title ⚙️ Instalación de Dependencias Parte 2/2

import os
import pandas as pd
import numpy as np
import subprocess
import sys
import time
import threading

def instalar_paquete_silencioso(comando, descripcion="Instalando dependencias"):
    """
    Ejecuta un comando de terminal en segundo plano suprimiendo la salida estándar
    y muestra una animación de transición de emojis en la celda de Colab.
    """
    # Variables de control para el hilo
    proceso_terminado = False
    emojis_animacion = ["⏳", "⌛"] # Emojis de transición
    emoji_exito = "✅"
    emoji_error = "❌"

    # Función que se ejecuta en el hilo paralelo
    def mostrar_progreso():
        i = 0
        while not proceso_terminado:
            # El retorno de carro '\r' sobrescribe la misma línea
            sys.stdout.write(f"\r{emojis_animacion[i % len(emojis_animacion)]} {descripcion}...")
            sys.stdout.flush()
            time.sleep(0.4)
            i += 1

    # Iniciar la animación en un hilo separado
    hilo_animacion = threading.Thread(target=mostrar_progreso)
    hilo_animacion.start()

    # Ejecutar el comando de instalación bloqueando la salida de consola
    resultado = subprocess.run(
        comando,
        shell=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

    # Detener el hilo de animación
    proceso_terminado = True
    hilo_animacion.join()

    # Evaluar el código de salida del subproceso y mostrar el estado final
    if resultado.returncode == 0:
        sys.stdout.write(f"\r{emoji_exito} {descripcion}... ¡✅ Completado exitosamente! \n")
    else:
        sys.stdout.write(f"\r{emoji_error} {descripcion}... ¡❌ Fallo en la ejecución!         \n")

if pd.__version__ != '1.5.3' or np.__version__ != '1.26.4':
    print("\n⚠️ Versiones desalineadas. Reiniciando el entorno para aplicar cambios...")
    os.kill(os.getpid(), 9)
else:
    print("\n✅ Entorno listo con Pandas 1.5.3 y NumPy 1.26.4")

# instalar_paquete_silencioso(
#     comando="pip install transformers==4.41.0 openai nltk opensmile noisereduce torch torchaudio pydub "thefuzz[speedup]" -q",
#     descripcion="Instalando Librerias"
# )

!pip -q install transformers==4.41.0 openai nltk torch torchaudio pydub "thefuzz[speedup]" opensmile noisereduce


import os
import json
import re
import unicodedata
import time
import torch
import nltk
from openai import OpenAI
from thefuzz import fuzz
from pydub import AudioSegment
from pydub.silence import split_on_silence
import noisereduce as nr
import soundfile as sf
from IPython.display import Audio, display
import opensmile
import matplotlib.pyplot as plt
from TTS.api import TTS

nltk.download('punkt', quiet=True)

print("✅ Librerías instaladas")

In [ ]:
#!pip install nvidia-cublas-cu12==12.4.5.8 nvidia-cuda-cupti-cu12==12.4.127 nvidia-cuda-nvrtc-cu12==12.4.127 nvidia-cuda-runtime-cu12==12.4.127 nvidia-cudnn-cu12==9.1.0.70 nvidia-cufft-cu12==11.2.1.3 nvidia-curand-cu12==10.3.5.147 nvidia-cusolver-cu12==11.6.1.9 nvidia-cusparse-cu12==12.3.1.170 nvidia-nvjitlink-cu12==12.4.127 -q


In [ ]:
# @title Definicion de funciones

# ==========================
# Definicion de funciones
# ==========================

def limpiar_texto_de_metadata(texto_completo):
    match = re.search(r"\s-\s[A-ZÁÉÍÓÚÑ]", texto_completo)
    if match:
        return texto_completo[:match.start()].strip()
    return texto_completo.strip()

def split_text_into_chunks(text, max_chars=800):
    if len(text) <= max_chars:
        return [text]
    sentences = nltk.tokenize.sent_tokenize(text, language='spanish')
    chunks = []
    current_chunk = ""
    for sentence in sentences:
        if len(current_chunk) + len(sentence) + 1 > max_chars:
            if current_chunk:
                chunks.append(current_chunk.strip())
            current_chunk = sentence
        else:
            current_chunk += (" " + sentence) if current_chunk else sentence
    if current_chunk:
        chunks.append(current_chunk.strip())
    return chunks

def normalizar_para_comparacion(texto):
    texto = ''.join(c for c in unicodedata.normalize('NFD', texto.lower()) if c.isalnum() or c.isspace()).strip()
    return " ".join(texto.split())

def verificar_audio_flexible(ruta_audio, texto_original, umbral_similitud=90):
    if not client:
        return False
    try:
        with open(ruta_audio, "rb") as audio_file:
            transcription = client.audio.transcriptions.create(model="whisper-1", file=audio_file, language="es")
        texto_transcrito = transcription.text
        if not texto_transcrito or len(texto_transcrito.strip()) < 5:
            print("    ⚠️ Verificación fallida: Whisper no detectó texto (posible ruido).")
            return False
        original_norm = normalizar_para_comparacion(texto_original)
        transcrito_norm = normalizar_para_comparacion(texto_transcrito)
        ratio = fuzz.ratio(original_norm, transcrito_norm)
        print(f"    -> Similitud del texto: {ratio}% (Umbral: {umbral_similitud}%%)")
        if ratio >= umbral_similitud:
            return True
        else:
            print("    ⚠️ Verificación fallida: Baja similitud.")
            return False
    except Exception as e:
        print(f"    ❌ Error durante la verificación con Whisper: {e}")
        return False

def limpiar_audio_generado(ruta_entrada, ruta_salida):
    try:
        data, rate = sf.read(ruta_entrada)
        ruido_reducido_data = nr.reduce_noise(y=data, sr=rate)
        ruta_temp = "/content/temp_audio_limpio.wav"
        sf.write(ruta_temp, ruido_reducido_data, rate)
        audio = AudioSegment.from_wav(ruta_temp)
        chunks = split_on_silence(audio, min_silence_len=500, silence_thresh=audio.dBFS-16)
        audio_final = sum(chunks, AudioSegment.empty()) if chunks else audio
        audio_final.export(ruta_salida, format="wav")
        if os.path.exists(ruta_temp):
            os.remove(ruta_temp)
        return True
    except Exception as e:
        print(f"    ❌ Error durante la limpieza del audio: {e}")
        return False

def limpiar_texto_summary(texto_completo):
    delimitadores = ["- Emoción complementaria utilizada:", "Mecanismos de atención:", "Mecanismo de atención:", "Resumen:", "Número total de caracteres:", "**"]
    primera_posicion = -1
    for d in delimitadores:
        posicion_actual = texto_completo.find(d)
        if posicion_actual != -1 and (primera_posicion == -1 or posicion_actual < primera_posicion):
            primera_posicion = posicion_actual
    if primera_posicion != -1:
        texto_completo = texto_completo[:primera_posicion]
    texto_limpio = texto_completo.replace('\n', ' ').replace('*', '').replace('-', '').strip()
    return re.sub(r'\s+', ' ', texto_limpio)

def encontrar_archivo_emocion(ruta_carpeta_locutor, emocion):
    termino_busqueda = "".join(c for c in unicodedata.normalize('NFD', emocion.lower()) if unicodedata.category(c) != 'Mn')
    if not os.path.exists(ruta_carpeta_locutor): return None
    for nombre_archivo in os.listdir(ruta_carpeta_locutor):
        if nombre_archivo.lower().endswith('.wav'):
            nombre_norm = "".join(c for c in unicodedata.normalize('NFD', nombre_archivo.lower()) if unicodedata.category(c) != 'Mn')
            if termino_busqueda in nombre_norm:
                return os.path.join(ruta_carpeta_locutor, nombre_archivo)
    return None

# Note: Duplicate function definition from source cell maintained as per subtask instructions
def split_text_into_chunks(text, max_chars=800):
    try: nltk.data.find('tokenizers/punkt')
    except LookupError: nltk.download('punkt', quiet=True)
    if len(text) <= max_chars: return [text]
    sentences = nltk.tokenize.sent_tokenize(text, language='spanish')
    chunks, current_chunk = [], ""
    for sentence in sentences:
        if len(current_chunk) + len(sentence) + 1 > max_chars:
            if current_chunk: chunks.append(current_chunk.strip())
            current_chunk = sentence
        else:
            current_chunk += (" " + sentence) if current_chunk else sentence
    if current_chunk: chunks.append(current_chunk.strip())
    return chunks

def limpiar_audio_generado(ruta_entrada, ruta_salida):
    """
    Limpia el ruido de fondo y recorta los silencios al inicio y al final de un archivo de audio.
    """
    try:
        # 1. Leer el audio
        data, rate = sf.read(ruta_entrada)

        # 2. Reducir ruido de fondo (asegurándose de que sea mono)
        if data.ndim > 1: data = data.mean(axis=1)
        ruido_reducido_data = nr.reduce_noise(y=data, sr=rate, prop_decrease=0.8)

        # Guardar temporalmente el archivo sin ruido
        ruta_temp_nr = os.path.join(LOCAL_TEMP_DIR, "temp_noise_reduced.wav")
        sf.write(ruta_temp_nr, ruido_reducido_data, rate)

        # 3. Cargar con Pydub para recortar silencios
        audio = AudioSegment.from_wav(ruta_temp_nr)
        silence_threshold = audio.dBFS - 16 # Umbral de silencio dinámico

        # Recortar silencio al inicio
        trim_leading = detect_leading_silence(audio, silence_threshold=silence_threshold)
        audio_trimmed_start = audio[trim_leading:]

        # Recortar silencio al final (invirtiendo el audio)
        trim_trailing = detect_leading_silence(audio_trimmed_start.reverse(), silence_threshold=silence_threshold)
        audio_trimmed_final = audio_trimmed_start.reverse()[trim_trailing:].reverse()

        # 4. Aplicar un suave fade in/out para un acabado profesional
        audio_final = audio_trimmed_final.fade_in(30).fade_out(30) # 30 milisegundos

        # 5. Exportar el archivo final y limpiar el temporal
        audio_final.export(ruta_salida, format="wav")
        if os.path.exists(ruta_temp_nr): os.remove(ruta_temp_nr)

        return True
    except Exception as e:
        tqdm.write(f"    ❌ Error durante la limpieza del audio: {e}")
        return False

print("✅ Funciones definidas")

In [ ]:
# @title --

target_pandas = '1.5.3'
target_numpy = '1.26.4'

current_pandas = pd.__version__
current_numpy = np.__version__

print(f'📊 Versión actual de Pandas: {current_pandas}')
print(f'📊 Versión actual de NumPy: {current_numpy}')

if current_pandas != target_pandas or current_numpy != target_numpy:
    print(f'\n⚠️ Versiones no coinciden. Instalando Pandas {target_pandas} y NumPy {target_numpy}...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', f'pandas=={target_pandas}', f'numpy=={target_numpy}', '--force-reinstall'])
    print('\n✅ Instalación completada. POR FAVOR, REINICIA EL ENTORNO DE EJECUCIÓN (Runtime > Restart runtime).')
else:
    print('\n✅ Las versiones ya son correctas')

# !pip install nvidia-cublas-cu12==12.4.5.8 nvidia-cuda-cupti-cu12==12.4.127 nvidia-cuda-nvrtc-cu12==12.4.127 nvidia-cuda-runtime-cu12==12.4.127 nvidia-cudnn-cu12==9.1.0.70 nvidia-cufft-cu12==11.2.1.3 nvidia-curand-cu12==10.3.5.147 nvidia-cusolver-cu12==11.6.1.9 nvidia-cusparse-cu12==12.3.1.170 nvidia-nvjitlink-cu12==12.4.127 -q
# print('✅ CUDA dependencies aligned with Torch 2.6.0')

# !pip -q install transformers==4.41.0 openai nltk torchaudio pydub "thefuzz[speedup]" opensmile noisereduce

## 2. Inicialización de Modelos y Rutas

Esta sección configura el proyecto. Aquí se realizan las siguientes acciones:

* **Montar Google Drive**: Se conecta el notebook al almacenamiento de Google Drive para acceder a los archivos del proyecto.
* **Definir Estructura de Carpetas**: Se establecen las rutas de entrada (videos, JSON) y de salida (audios generados, audios procesados, evaluaciones y mezcla final).
* **Cargar Modelo TTS**: Se inicializa y carga en memoria el modelo `XTTS v2` de Coqui, preparándolo para la generación de voz.
* **Configurar Cliente OpenAI**: Se establece la conexión con la API de OpenAI, necesaria para la verificación de audio con Whisper.

In [ ]:
# @title ⚙️ API de Google Drive, TTS y OpenAI

import os
import io
import unicodedata
import warnings
from IPython.display import clear_output
from IPython.utils import io as ipy_io

warnings.filterwarnings("ignore", category=FutureWarning)

# =========================================================
# 1. Instalación e Importación de Dependencias (Silencioso)
# =========================================================
with ipy_io.capture_output() as captured:
    !pip install -q --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib

from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
from google.colab import drive
import torch
from TTS.api import TTS
from openai import OpenAI

# =========================================================
# 2. Configuración de Credenciales
# =========================================================
CLIENT_ID = os.environ.get("CLIENT_ID", "")  # credencial externa
CLIENT_SECRET = os.environ.get("CLIENT_SECRET", "")  # credencial externa
REFRESH_TOKEN = os.environ.get("REFRESH_TOKEN", "")  # credencial externa
SCOPES = ['https://www.googleapis.com/auth/drive']

def autenticar_drive():
    try:
        creds = Credentials(None, refresh_token=REFRESH_TOKEN, token_uri='https://oauth2.googleapis.com/token', client_id=CLIENT_ID, client_secret=CLIENT_SECRET, scopes=SCOPES)
        creds.refresh(Request())
        return build('drive', 'v3', credentials=creds), "✅ Conectado"
    except Exception as e:
        return None, f"❌ Error de autenticación: {e}"

# Montaje nativo de Colab (Silencioso)
with ipy_io.capture_output() as captured:
    drive.mount('/content/drive', force_remount=True)

drive_service, estado_drive = autenticar_drive()

# =========================================================
# 3. Input del Usuario
# =========================================================
print("📁 CONFIGURACIÓN DE CARPETA BASE DEL PROYECTO")
FOLDER_ID_BASE_PROYECTO = input("🔗 Ingresa el ID de la carpeta base del proyecto: ").strip()

while not FOLDER_ID_BASE_PROYECTO:
    print("⚠️ El ID no puede estar vacío.")
    FOLDER_ID_BASE_PROYECTO = input("🔗 Ingresa el ID de la carpeta base del proyecto: ").strip()

FOLDER_ID_EMOCIONES_BASE = "16WAuMzjmC4xNPGbr3QWIYHspKBl4H2s7"
os.makedirs("temp_drive_files", exist_ok=True)
os.makedirs("temp_output", exist_ok=True)
LOCAL_TEMP_DIR = "temp_drive_files"
LOCAL_OUTPUT_DIR = "temp_output"

# =========================================================
# 4. Resolución de Directorios
# =========================================================
def crear_o_obtener_carpeta(service, nombre, id_padre):
    query = f"name='{nombre}' and mimeType='application/vnd.google-apps.folder' and '{id_padre}' in parents and trashed=false"
    items = service.files().list(q=query, fields="files(id)").execute().get('files', [])
    if items: return items[0].get('id')
    folder = service.files().create(body={'name': nombre, 'mimeType': 'application/vnd.google-apps.folder', 'parents': [id_padre]}, fields='id').execute()
    return folder.get('id')

if drive_service:
    FOLDER_ID_SEGMENTS = crear_o_obtener_carpeta(drive_service, "segments descriptions", FOLDER_ID_BASE_PROYECTO)
    FOLDER_ID_INPUT = crear_o_obtener_carpeta(drive_service, "Input", FOLDER_ID_SEGMENTS)
    FOLDER_ID_OUTPUT_MAIN = crear_o_obtener_carpeta(drive_service, "Output AD", FOLDER_ID_SEGMENTS)
    FOLDER_ID_AD = crear_o_obtener_carpeta(drive_service, "AD", FOLDER_ID_OUTPUT_MAIN)
    FOLDER_ID_EVALUACION = crear_o_obtener_carpeta(drive_service, "Evaluación prosódica", FOLDER_ID_OUTPUT_MAIN)
    FOLDER_ID_OUTPUT_PROCESADOS = FOLDER_ID_AD

def listar_archivos(service, folder_id):
    try: return service.files().list(q=f"'{folder_id}' in parents and trashed=false", fields="files(id, name, mimeType)").execute().get('files', [])
    except: return []

def descargar_archivo(service, file_id, local_path):
    try:
        request = service.files().get_media(fileId=file_id)
        fh = io.BytesIO()
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done: _, done = downloader.next_chunk()
        with open(local_path, 'wb') as f: f.write(fh.getvalue())
        return local_path
    except: return None

def subir_archivo(service, local_path, folder_id):
    try:
        media = MediaFileUpload(local_path, resumable=True)
        return service.files().create(body={'name': os.path.basename(local_path), 'parents': [folder_id]}, media_body=media, fields='id').execute().get('id')
    except: return None

def encontrar_y_descargar_referencia_drive(service, base_id, locutor, emocion):
    carpetas = listar_archivos(service, base_id)
    id_locutor = next((c['id'] for c in carpetas if c['name'] == locutor), None)
    if not id_locutor: return None
    termino = "".join(c for c in unicodedata.normalize('NFD', emocion.lower()) if unicodedata.category(c) != 'Mn')
    for arc in listar_archivos(service, id_locutor):
        norm = "".join(c for c in unicodedata.normalize('NFD', arc['name'].lower()) if unicodedata.category(c) != 'Mn')
        if termino in norm and arc['name'].lower().endswith('.wav'):
            ruta = os.path.join(LOCAL_TEMP_DIR, arc['name'])
            descargar_archivo(service, arc['id'], ruta)
            return ruta
    return None

# =========================================================
# 5. Inicialización de Modelos y API
# =========================================================
os.environ["COQUI_TOS_AGREED"] = "1"
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import XttsAudioConfig, XttsArgs
from TTS.config.shared_configs import BaseDatasetConfig

torch.serialization.add_safe_globals([XttsConfig, XttsAudioConfig, BaseDatasetConfig, XttsArgs])
device = "cuda" if torch.cuda.is_available() else "cpu"
_original_torch_load = torch.load
def parche_torch_load_inseguro(*args, **kwargs):
    kwargs['weights_only'] = False
    return _original_torch_load(*args, **kwargs)

#!echo "y" | python -c 'from TTS.api import TTS; TTS(model_name="tts_models/multilingual/multi-dataset/xtts_v2")'
try:
    # 3. Inyectamos el parche
    torch.load = parche_torch_load_inseguro

    # 4. Instanciamos el modelo acústico (ahora podrá leer el checkpoint)
    tts = TTS(model_name="tts_models/multilingual/multi-dataset/xtts_v2", progress_bar=False).to(device)
    print("✅ Motor TTS (XTTS v2) cargado exitosamente en VRAM.")

except Exception as e:
    print(f"❌ Error crítico al cargar el modelo TTS: {e}")

finally:
    # 5. Restauramos la función original para mantener la seguridad del resto del notebook
    torch.load = _original_torch_load

# Capturamos los prints internos del modelo TTS para no ensuciar la consola
with ipy_io.capture_output() as captured:
    tts = TTS(model_name="tts_models/multilingual/multi-dataset/xtts_v2").to(device)

estado_openai = "❌ Fallo en conexión"
try:
    client = OpenAI(api_key="REDACTED_USE_ENV")
    client.models.list()
    estado_openai = "✅ Autenticado"
except Exception as e:
    client = None
    estado_openai = f"❌ Error API: {e}"
# =========================================================
# 6. Interfaz de Salida Estructurada
# =========================================================
clear_output() # Limpia todo el historial de ejecución anterior a este punto

print("🎉 === INICIALIZACIÓN COMPLETADA EXITOSAMENTE ===")
print(f"🎯 Carpeta Base Objetivo: {FOLDER_ID_BASE_PROYECTO}")

print("\n📋 === RESUMEN DE TOPOLOGÍA EN DRIVE ===")
print(f"📂 Carpeta Base")
print(f"   └── 📂 segments descriptions (ID: {FOLDER_ID_SEGMENTS})")
print(f"       ├── 📂 Input (ID: {FOLDER_ID_INPUT})")
print(f"       └── 📂 Output AD (ID: {FOLDER_ID_OUTPUT_MAIN})")
print(f"           ├── 📂 AD (ID: {FOLDER_ID_AD})")
print(f"           └── 📂 Evaluación prosódica (ID: {FOLDER_ID_EVALUACION})")

print("\n🔗 === ENLACES DE ACCESO RÁPIDO ===")
print(f"   📥 Input: https://drive.google.com/drive/folders/{FOLDER_ID_INPUT}")
print(f"   📤 Output AD: https://drive.google.com/drive/folders/{FOLDER_ID_AD}")
print(f"   📊 Evaluación: https://drive.google.com/drive/folders/{FOLDER_ID_EVALUACION}")

print("\n⚙️ === ESTADO DE SERVICIOS Y MODELOS ===")
print(f"   ☁️  API Google Drive: {estado_drive}")
print(f"   🎙️  Motor TTS (XTTS v2): ✅ Cargado en hardware ({device.upper()})")
print(f"   🧠  API OpenAI: {estado_openai}")
print("===================================================")

## 3. Carga y Preparación de Tareas de Generación

El siguiente código escanea las carpetas de entrada definidas previamente. Busca los archivos `prosody_analysis.json` para extraer la **emoción principal** y los `video_summary.json` para obtener el **texto a sintetizar**.

Toda esta información se organiza en una tabla (DataFrame de Pandas) que resume las tareas de generación que se realizarán a continuación.

In [ ]:
# @title ⚙️ FASE 1: LECTURA Y PREPARACIÓN DE TAREAS DE GENERACION
# =========================================================
import io
import json
import os
import pandas as pd
from googleapiclient.http import MediaIoBaseDownload
from tqdm.notebook import tqdm # Usamos la versión para notebooks

tareas_de_generacion = []
print("--- 📥 Analizando la carpeta 'Input' en Drive y extrayendo tareas ---")

# Obtenemos la lista de carpetas de video
carpetas_de_videos = [
    item for item in listar_archivos(drive_service, FOLDER_ID_INPUT)
    if item['mimeType'] == 'application/vnd.google-apps.folder'
]

# Usamos tqdm para mostrar una barra de progreso en lugar de múltiples prints
for video_folder in tqdm(carpetas_de_videos, desc="🔎 Analizando carpetas de video"):
    video_id_folder = video_folder['name']
    folder_id = video_folder['id']

    try:
        archivos_en_carpeta = listar_archivos(drive_service, folder_id)
        summary_file = next((f for f in archivos_en_carpeta if f['name'] == 'video_summary.json'), None)
        prosody_file = next((f for f in archivos_en_carpeta if f['name'] == 'prosody_analysis.json'), None)

        if summary_file and prosody_file:
            local_summary_path = descargar_archivo(drive_service, summary_file['id'], os.path.join(LOCAL_TEMP_DIR, f"{video_id_folder}_summary.json"))
            local_prosody_path = descargar_archivo(drive_service, prosody_file['id'], os.path.join(LOCAL_TEMP_DIR, f"{video_id_folder}_prosody.json"))

            if local_summary_path and local_prosody_path:
                with open(local_summary_path, 'r', encoding='utf-8') as f: datos_summary = json.load(f)
                with open(local_prosody_path, 'r', encoding='utf-8') as f: datos_prosody = json.load(f)

                my_text = limpiar_texto_summary(datos_summary["main_summary"])
                tareas_de_generacion.append({
                    "Video ID": video_id_folder,
                    "Texto a Generar": my_text,
                    "Emoción": datos_prosody["complementary_emotion"]
                })
    except Exception as e:
        # tqdm.write se usa para imprimir mensajes sin romper la barra de progreso
        tqdm.write(f"⚠️ Error al procesar '{video_id_folder}': {e}")

if tareas_de_generacion:
    df_tareas = pd.DataFrame(tareas_de_generacion)
    print(f"\n✅ {len(tareas_de_generacion)} tareas cargadas exitosamente.")
    print("\n--- 📝 RESUMEN DE TAREAS A REALIZAR ---")

    # 🛠️ PARCHE: Imprimimos en texto plano para evitar el crash del frontend de Colab
    print(df_tareas.to_string(index=False))
else:
    print("\n❌ No se encontraron tareas válidas para procesar.")

In [ ]:
# @title
def verificar_audio_flexible(ruta_audio, texto_original, umbral_similitud=90):
    if not client:
        print("    ⚠️ Error de instancia: El cliente de OpenAI no está inicializado.")
        return False

    try:
        with open(ruta_audio, "rb") as audio_file:
            transcription = client.audio.transcriptions.create(
                model="whisper-1",
                file=audio_file,
                language="es",
                response_format="text"
            )

        # Extracción segura del texto dependiendo de la respuesta del SDK
        texto_transcrito = transcription if isinstance(transcription, str) else transcription.text

        if not texto_transcrito or len(texto_transcrito.strip()) < 5:
            print("    ⚠️ Verificación fallida: No se detectó señal de voz útil.")
            return False

        # Normalización Unicode (NFD) para homogeneizar la comparación (ignora tildes y mayúsculas)
        original_norm = " ".join("".join(c for c in unicodedata.normalize('NFD', texto_original.lower()) if c.isalnum() or c.isspace()).split())
        transcrito_norm = " ".join("".join(c for c in unicodedata.normalize('NFD', texto_transcrito.lower()) if c.isalnum() or c.isspace()).split())

        # Cálculo de similitud mediante Distancia de Levenshtein
        ratio = fuzz.ratio(original_norm, transcrito_norm)
        print(f"    -> Precisión (WER Inversa): {ratio}%. \n       Original: '{original_norm[:40]}...'\n       Inferido: '{transcrito_norm[:40]}...'")

        return ratio >= umbral_similitud

    except Exception as e:
        # Captura y log de errores HTTP o de parseo
        print(f"    ❌ Excepción de API (OpenAI): {type(e).__name__} - {str(e)}")
        return False
print("Funciones para transcripcion definidas")

In [ ]:
# @title
## @title 🛠️ Parche de Compatibilidad de Audio (Ejecutar ANTES de la Fase 2)

#!pip install torch==2.6.0
# import torch
# import torchaudio
# import soundfile as sf
# import TTS.tts.models.xtts as xtts_module

# def xtts_load_audio_safepatch(audiopath, sampling_rate):
#     """
#     Función de reemplazo (wrapper) que evita la dependencia de torchcodec
#     utilizando soundfile para la extracción del tensor acústico.
#     """
#     data, lsr = sf.read(audiopath)
#     audio = torch.tensor(data, dtype=torch.float32)

#     # Alineación de dimensiones
#     if audio.ndim == 1:
#         audio = audio.unsqueeze(0)
#     else:
#         audio = audio.t()

#     # Conversión a canal Mono
#     if audio.size(0) != 1:
#         audio = torch.mean(audio, dim=0, keepdim=True)

#     # Remuestreo a 24kHz
#     if lsr != sampling_rate:
#         audio = torchaudio.functional.resample(audio, orig_freq=lsr, new_freq=sampling_rate)

#     # Normalización
#     if not torch.any(audio < 0):
#         audio = (audio / audio.max()) * 2 - 1

#     return audio

# # Inyección directa en el módulo global de XTTS
# xtts_module.load_audio = xtts_load_audio_safepatch

# print("✅ Parche de compatibilidad aplicado exitosamente en la memoria.")
# print("▶️ Ya puedes ejecutar tu celda de la Fase 2 con total seguridad.")

In [ ]:
# @title
import noisereduce as nr
import soundfile as sf
from pydub import AudioSegment
from pydub.silence import detect_leading_silence
from tqdm.notebook import tqdm

def limpiar_audio_generado(ruta_entrada, ruta_salida):
    """
    Limpia el ruido de fondo y recorta los silencios al inicio y al final de un archivo de audio.
    """
    try:
        # 1. Leer el audio
        data, rate = sf.read(ruta_entrada)

        # 2. Reducir ruido de fondo (asegurándose de que sea mono)
        if data.ndim > 1: data = data.mean(axis=1)
        ruido_reducido_data = nr.reduce_noise(y=data, sr=rate, prop_decrease=0.8)

        # Guardar temporalmente el archivo sin ruido
        ruta_temp_nr = os.path.join(LOCAL_TEMP_DIR, "temp_noise_reduced.wav")
        sf.write(ruta_temp_nr, ruido_reducido_data, rate)

        # 3. Cargar con Pydub para recortar silencios
        audio = AudioSegment.from_wav(ruta_temp_nr)
        silence_threshold = audio.dBFS - 16 # Umbral de silencio dinámico

        # Recortar silencio al inicio
        trim_leading = detect_leading_silence(audio, silence_threshold=silence_threshold)
        audio_trimmed_start = audio[trim_leading:]

        # Recortar silencio al final (invirtiendo el audio)
        trim_trailing = detect_leading_silence(audio_trimmed_start.reverse(), silence_threshold=silence_threshold)
        audio_trimmed_final = audio_trimmed_start.reverse()[trim_trailing:].reverse()

        # 4. Aplicar un suave fade in/out para un acabado profesional
        audio_final = audio_trimmed_final.fade_in(30).fade_out(30) # 30 milisegundos

        # 5. Exportar el archivo final y limpiar el temporal
        audio_final.export(ruta_salida, format="wav")
        if os.path.exists(ruta_temp_nr): os.remove(ruta_temp_nr)

        return True
    except Exception as e:
        tqdm.write(f"    ❌ Error durante la limpieza del audio: {e}")
        return False
# =========================================================
# --- 🔊 FASE 2: BUCLE PRINCIPAL DE GENERACIÓN DE AUDIO (VISUALIZACIÓN MEJORADA) ---
# =========================================================

locutores = ["Andrea", "Sergio", "Sebastian"]
max_intentos_por_chunk = 3

if not tareas_de_generacion:
    print("\n❌ La lista de tareas está vacía. No se generará ningún audio.")
else:
    for tarea in tqdm(tareas_de_generacion, desc="▶️  Progreso Total de Generación"):
        video_id, texto_a_generar, emocion_a_buscar = tarea["Video ID"], tarea["Texto a Generar"], tarea["Emoción"]

        tqdm.write("\n" + "="*80)
        tqdm.write(f"📄 TAREA | VIDEO: {video_id} | EMOCIÓN: {emocion_a_buscar}")
        tqdm.write("="*80)

        chunks = split_text_into_chunks(texto_a_generar)

        for locutor in tqdm(locutores, desc=f"   🎤 Locutores para {video_id}", leave=False):
            tqdm.write(f"\n🎤 PROCESANDO LOCUTOR: {locutor.upper()}")
            tqdm.write("-" * 60)

            ruta_referencia_voz_local = encontrar_y_descargar_referencia_drive(drive_service, FOLDER_ID_EMOCIONES_BASE, locutor, emocion_a_buscar)

            if not ruta_referencia_voz_local:
                tqdm.write(f"  ❌ No se encontró audio de referencia. Saltando.")
                continue

            tqdm.write(f"  🎵 Usando referencia: '{os.path.basename(ruta_referencia_voz_local)}'")
            temp_audio_files = []

            for i, chunk in enumerate(chunks):
                tqdm.write(f"  📄 Fragmento [{i+1}/{len(chunks)}]")
                intentos, verificado = 0, False
                ruta_temp_chunk = os.path.join(LOCAL_OUTPUT_DIR, f"temp_{video_id}_{locutor}_{i}.wav")

                while intentos < max_intentos_por_chunk:
                    intentos += 1
                    tqdm.write(f"    - Intento {intentos}/{max_intentos_por_chunk}:")
                    tqdm.write(f"      ⚙️ Generando audio...")

                    # Se asume que la librería TTS puede ser muy verbosa.
                    # El ocultar su salida interna se maneja en la configuración de la librería si es posible.
                    tts.tts_to_file(text=chunk.replace('.', ''), speaker_wav=ruta_referencia_voz_local, language="es", file_path=ruta_temp_chunk)

                    tqdm.write(f"      🔍 Verificando...")
                    verificado = verificar_audio_flexible(ruta_temp_chunk, chunk) # Asumimos que esta función también imprime su resultado de forma concisa

                    if verificado:
                        tqdm.write(f"        ✅ Verificación exitosa.")
                        break

                if not verificado:
                    tqdm.write(f"        ⚠️  Verificación fallida. Se usará el último audio generado.")

                temp_audio_files.append(ruta_temp_chunk)

            if temp_audio_files:
                tqdm.write(f"\n  ✨ Procesando y subiendo el audio final...")
                combined_audio = sum([AudioSegment.from_wav(f) for f in temp_audio_files], AudioSegment.empty())
                nombre_final = f"{video_id}_{locutor}_{emocion_a_buscar}.wav"
                ruta_local_procesado = os.path.join(LOCAL_OUTPUT_DIR, nombre_final)

                # Se crea un archivo intermedio antes de limpiar
                ruta_local_intermedio = os.path.join(LOCAL_OUTPUT_DIR, f"intermedio_{nombre_final}")
                combined_audio.export(ruta_local_intermedio, format="wav")

                # --- LLAMADA A LA NUEVA FUNCIÓN DE LIMPIEZA ---
                tqdm.write("    🧼 Aplicando limpieza (reducción de ruido y recorte de silencios)...")
                if limpiar_audio_generado(ruta_local_intermedio, ruta_local_procesado):
                    subir_archivo(drive_service, ruta_local_procesado, FOLDER_ID_AD) # Asumo que FOLDER_ID_AD es el destino
                    tqdm.write(f"    📤 Archivo '{nombre_final}' limpio y subido a Drive.")
                else:
                    tqdm.write(f"    ⚠️ Falló la limpieza, se subirá el archivo sin procesar.")
                    subir_archivo(drive_service, ruta_local_intermedio, FOLDER_ID_AD)

            # Limpieza de archivos temporales
            archivos_a_limpiar = temp_audio_files + [ruta_referencia_voz_local]
            if temp_audio_files and 'ruta_local_intermedio' in locals():
                archivos_a_limpiar.extend([ruta_local_intermedio, ruta_local_procesado])

            for temp_file in archivos_a_limpiar:
                if temp_file and os.path.exists(temp_file):
                    os.remove(temp_file)

    print("\n\n" + "#"*80)
    print("###      PROCESO COMPLETO DE GENERACIÓN FINALIZADO      ###")
    print("#"*80)


## 7. Evaluación Prosódica de los Audios Generados

Una vez que todos los audios han sido generados, procesados y guardados en la carpeta `Audio Procesado Final`, esta sección final se encarga de realizar un análisis objetivo de sus características prosódicas utilizando la librería `openSMILE`.

**Proceso de Evaluación:**
1.  **Análisis con openSMILE:** Se extraen 88 características de cada audio utilizando el set de features `eGeMAPSv02`.
2.  **Selección y Normalización:** Se seleccionan 4 características clave (Intensidad, Entonación, Ritmo, Duración) y se normalizan en una escala del 1 al 5 para poder compararlas.
3.  **Generación de Gráficas:** Se crean y guardan varias visualizaciones:
    * Promedio por emoción.
    * Promedio por video.
    * Comparativa de locutores para cada video.
    * Gráficas individuales para cada archivo de audio.

In [ ]:
!pip install opensmile

In [ ]:
import opensmile
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import csv # <--- Librería nativa de Python
from tqdm.notebook import tqdm

# =========================================================
# 🛠️ BYPASS DEFINITIVO PARA EXPORTACIÓN DE CSV
# =========================================================
def exportar_csv_seguro(df, ruta_salida):
    """
    Exporta un DataFrame a CSV usando Python puro.
    Evita los choques de versiones de Pandas en memoria al ignorar la función to_csv().
    """
    # 1. Aplanar índices complejos (como los que genera openSMILE)
    df_plano = df.reset_index() if isinstance(df.index, pd.MultiIndex) else df.copy()

    # 2. Convertir todo a cadenas de texto para evitar conflictos de numpy
    df_str = df_plano.astype(str)

    # 3. Escribir el archivo manualmente
    with open(ruta_salida, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(df_str.columns) # Cabeceras
        writer.writerows(df_str.values) # Datos
# =========================================================

# --- 1. FUNCIÓN DE ANÁLISIS (VISUALIZACIÓN MEJORADA) ---

def analizar_prosodia_opensmile(service, id_carpeta_audios, id_carpeta_salida):
    print("\n--- 🔎 FASE 1: Análisis de Audio con openSMILE ---")

    archivos_en_drive = listar_archivos(service, id_carpeta_audios)
    archivos_wav_drive = [f for f in archivos_en_drive if f['name'].lower().endswith(".wav")]

    if not archivos_wav_drive:
        print("  ⚠️ No se encontraron archivos .wav en la carpeta de Drive. No se puede continuar.")
        return None

    rutas_locales_wav = []
    for archivo_drive in tqdm(archivos_wav_drive, desc="  📥 Descargando audios"):
        ruta_local = os.path.join(LOCAL_TEMP_DIR, archivo_drive['name'])
        descargar_archivo(service, archivo_drive['id'], ruta_local)
        rutas_locales_wav.append(ruta_local)

    print(f"  ⚙️  Procesando {len(rutas_locales_wav)} archivos con openSMILE...")
    smile = opensmile.Smile(
        feature_set=opensmile.FeatureSet.eGeMAPSv02,
        feature_level=opensmile.FeatureLevel.Functionals
    )
    df_features_full = smile.process_files(rutas_locales_wav)

    print("  📤 Subiendo reporte completo a Drive...")
    ruta_csv_completo_local = os.path.join(LOCAL_OUTPUT_DIR, "Analisis_openSMILE_Completo.csv")

    # Aplicamos nuestro Bypass aquí
    exportar_csv_seguro(df_features_full, ruta_csv_completo_local)
    subir_archivo(service, ruta_csv_completo_local, id_carpeta_salida)

    # --- Procesamiento de datos ---
    feature_mapping = {
        'Intensidad': 'loudness_sma3_amean', 'Entonación': 'F0semitoneFrom27.5Hz_sma3nz_amean',
        'Ritmo': 'VoicedSegmentsPerSec', 'Duración': 'MeanUnvoicedSegmentLength'
    }
    df_selected = df_features_full[list(feature_mapping.values())].copy()
    df_selected.columns = list(feature_mapping.keys())

    def normalize_zscore_1_5(df, columns):
        df_norm = df.copy()
        for col in columns:
            mean, std = df_norm[col].mean(), df_norm[col].std()
            if std == 0 or pd.isna(std): df_norm[col] = 3; continue
            z = (df_norm[col] - mean) / std
            min_z, max_z = z.min(), z.max()
            df_norm[col] = 1 + 4 * (z - min_z) / (max_z - min_z) if max_z != min_z else 3
            df_norm[col] = df_norm[col].clip(1, 5)
        return df_norm

    df_normalized = normalize_zscore_1_5(df_selected, list(feature_mapping.keys()))
    df_normalized.reset_index(inplace=True)
    df_normalized['Archivo'] = df_normalized['file'].apply(lambda f: os.path.basename(f))

    regex_pattern = r'^(?P<Video_ID>.+?)_(?P<Locutor>Andrea|Sergio|Sebastian)_(?P<Emocion>.+)\.wav$'
    df_parsed = df_normalized['Archivo'].str.extract(regex_pattern)
    df_normalized = pd.concat([df_normalized, df_parsed], axis=1)

    correcciones = {"extasis": "Éxtasis", "aceptacion": "Aceptación", "admiracion": "Admiración", "aprension": "Aprensión", "distraccion": "Distracción", "asombro": "Asombro", "pensamientoprofundo": "Pensamiento profundo", "duelo": "Duelo", "aburrimiento": "Aburrimiento", "aversion": "Aversión", "molestia": "Molestia", "furia": "Furia", "interes": "Interés", "vigilancia": "Vigilancia"}
    df_normalized["Emocion"] = df_normalized["Emocion"].str.replace(' ', '').str.lower().map(correcciones).fillna(df_normalized["Emocion"])

    print("  ✅ Análisis y selección de características finalizado.")
    return df_normalized

# --- 2. LÓGICA DE GRÁFICAS ---

def _crear_grafica_radar(ax, angles, features, valores, titulo, color, etiqueta=''):
    ax.plot(angles, valores, 'o-', linewidth=2, color=color, label=etiqueta)
    ax.fill(angles, valores, alpha=0.2, color=color)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(features, fontsize=8)
    ax.set_yticks(range(1, 6))
    ax.set_ylim(0, 5)
    ax.set_title(titulo, y=1.1, fontdict={'fontsize': 12, 'fontweight': 'bold'})

def graficar_y_subir(fig, service, nombre_archivo, id_carpeta_salida):
    ruta_local = os.path.join(LOCAL_OUTPUT_DIR, nombre_archivo)
    fig.savefig(ruta_local, dpi=150)
    plt.close(fig)
    subir_archivo(service, ruta_local, id_carpeta_salida)

def graficar_comparativa_emociones(df_norm, angles, features):
    df_promedio = df_norm.groupby("Emocion")[features].mean()
    fig, ax = plt.subplots(figsize=(10, 8), subplot_kw=dict(polar=True))
    colormap = plt.cm.get_cmap('gist_rainbow', len(df_promedio))
    for i, (emocion, row) in enumerate(df_promedio.iterrows()):
        valores = row.tolist() + [row[0]]
        _crear_grafica_radar(ax, angles, features, valores, "Análisis Promedio por Emoción", colormap(i), etiqueta=emocion)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    plt.tight_layout()
    return fig

def graficar_promedio_por_video(df_norm, angles, features):
    df_promedio = df_norm.groupby("Video_ID")[features].mean()
    num_videos = len(df_promedio)
    num_cols = min(num_videos, 4); num_rows = (num_videos + num_cols - 1) // num_cols
    fig, axes = plt.subplots(figsize=(20, 5 * num_rows), nrows=num_rows, ncols=num_cols, subplot_kw=dict(polar=True))
    axes = axes.flatten()
    for i, (video_id, row) in enumerate(df_promedio.iterrows()):
        valores = row.tolist() + [row[0]]
        _crear_grafica_radar(axes[i], angles, features, valores, video_id, "darkorange")
    for j in range(i + 1, len(axes)): fig.delaxes(axes[j])
    fig.suptitle("Análisis Prosódico Promedio por Video", fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    return fig

def graficar_comparativa_locutores(df_norm, angles, features):
    videos = df_norm["Video_ID"].unique()
    num_videos = len(videos)
    num_cols = min(num_videos, 4); num_rows = (num_videos + num_cols - 1) // num_cols
    colores_locutor = {'Andrea': 'red', 'Sergio': 'blue', 'Sebastian': 'green'}
    fig, axes = plt.subplots(figsize=(20, 5 * num_rows), nrows=num_rows, ncols=num_cols, subplot_kw=dict(polar=True))
    axes = axes.flatten()
    for i, video_id in enumerate(videos):
        ax = axes[i]
        df_video = df_norm[df_norm["Video_ID"] == video_id]
        for locutor, color in colores_locutor.items():
            df_locutor = df_video[df_video["Locutor"] == locutor]
            if not df_locutor.empty:
                valores = df_locutor[features].iloc[0].tolist() + [df_locutor[features].iloc[0, 0]]
                _crear_grafica_radar(ax, angles, features, valores, video_id, color, etiqueta=locutor)
        ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize='small')
    for j in range(i + 1, len(axes)): fig.delaxes(axes[j])
    fig.suptitle("Comparativa de Locutores por Video", fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    return fig

# --- 3. FUNCIÓN PRINCIPAL DE EJECUCIÓN ---

def ejecutar_analisis_completo(service, id_carpeta_audios, id_carpeta_evaluacion):
    print("\n" + "="*80)
    print("###      📊 INICIANDO FASE DE EVALUACIÓN PROSÓDICA      ###")
    print("="*80)

    df_analisis = analizar_prosodia_opensmile(service, id_carpeta_audios, id_carpeta_evaluacion)

    if df_analisis is None or df_analisis.empty:
        print("\n❌ No se generaron datos de análisis. El proceso de graficación se omitirá.")
    else:
        print("\n--- 🎨 FASE 2: Generación de Gráficas ---")
        columnas = ["Intensidad", "Entonación", "Ritmo", "Duración"]
        angles = np.linspace(0, 2 * np.pi, len(columnas), endpoint=False).tolist() + [0]

        print("  1. Creando Gráfica Comparativa de Emociones...")
        fig1 = graficar_comparativa_emociones(df_analisis, angles, columnas)
        graficar_y_subir(fig1, service, "Grafica_Comparativa_Emociones.png", id_carpeta_evaluacion)

        print("  2. Creando Gráfica de Promedios por Video...")
        fig2 = graficar_promedio_por_video(df_analisis, angles, columnas)
        graficar_y_subir(fig2, service, "Grafica_Promedio_Videos.png", id_carpeta_evaluacion)

        print("  3. Creando Gráfica Comparativa de Locutores...")
        fig3 = graficar_comparativa_locutores(df_analisis, angles, columnas)
        graficar_y_subir(fig3, service, "Grafica_Comparativa_Locutores.png", id_carpeta_evaluacion)

        print("\n  📤 Subiendo reporte CSV final...")
        ruta_csv_final = os.path.join(LOCAL_OUTPUT_DIR, "Analisis_Prosodico_Normalizado.csv")

        # Aplicamos nuestro Bypass al final también
        exportar_csv_seguro(df_analisis, ruta_csv_final)

        subir_archivo(service, ruta_csv_final, id_carpeta_evaluacion)
        print("  ✅ Gráficas y reportes subidos a Drive.")

    # Limpieza de archivos temporales
    print("\n--- 🧹 FASE 3: Limpieza de Archivos Temporales ---")
    for folder in [LOCAL_TEMP_DIR, LOCAL_OUTPUT_DIR]:
        for filename in os.listdir(folder):
            if os.path.isfile(os.path.join(folder, filename)):
                os.remove(os.path.join(folder, filename))
    print("  ✅ Limpieza completada.")

    print("\n" + "#"*80)
    print("###      EVALUACIÓN PROSÓDICA FINALIZADA      ###")
    print("#"*80)

ejecutar_analisis_completo(
    service=drive_service,
    id_carpeta_audios=FOLDER_ID_AD,
    id_carpeta_evaluacion=FOLDER_ID_EVALUACION
)

## 5. Ducking y Sincronización

El siguiente script automatiza la post-producción de la pista de audio final. El proceso es el siguiente:

1.  **Carga los Archivos**: Toma el video original, el archivo `JSON` con los tiempos precisos de cada diálogo y la carpeta con todos los audios generados en el paso anterior.
2.  **Extrae el Audio Base**: Obtiene la pista de audio original del archivo de video.
3.  **Aplica Ducking**: Atenúa automáticamente el volumen del audio base durante los intervalos de tiempo especificados en el `JSON`, creando espacio para los diálogos. Se pueden ajustar parámetros como la cantidad de reducción y la duración del "fade" para un acabado profesional.
4.  **Sincroniza y Mezcla**: Carga cada audio de diálogo y lo inserta en la pista base atenuada en el momento exacto que indica el `JSON`.
5.  **Exporta los Resultados**: Genera
    * Un archivo de audio (`.mp3`) con la mezcla completa.

In [ ]:
instalar_paquete_silencioso(
    comando="pip install -q -U demucs",
    descripcion="Demucs"
)

In [ ]:
# @title
from tqdm.notebook import tqdm
import json
import os
import re
import subprocess
import imageio_ffmpeg
import io
import glob
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from pydub import AudioSegment
from pydub.silence import detect_leading_silence
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
import noisereduce as nr # <--- 1. IMPORTACIÓN AÑADIDA

# --- FUNCIONES AUXILIARES "SILENCIOSAS" ---

def listar_archivos(service, folder_id):
    # (Función sin cambios)
    query = f"'{folder_id}' in parents and trashed=false"
    try:
        request = service.files().list(q=query, fields="files(id, name, mimeType)")
        resp, content = request.http.request(request.uri, method=request.method, body=request.body, headers=request.headers)
        decoded_content = content.decode('utf-8', errors='replace')
        data = json.loads(decoded_content)
        return data.get('files', [])
    except Exception as e:
        print(f"Error al listar archivos en la carpeta {folder_id}: {e}")
        return []

def descargar_archivo(service, file_id, local_path):
    # (Función sin cambios)
    try:
        request = service.files().get_media(fileId=file_id)
        fh = io.BytesIO()
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()
        with open(local_path, 'wb') as f:
            f.write(fh.getvalue())
        return local_path
    except Exception as e:
        print(f"Error al descargar el archivo {file_id}: {e}")
        return None

def find_file_in_drive(service, folder_id, file_pattern):
    # (Visualización mejorada: sin flecha)
    files = listar_archivos(service, folder_id)
    for f in files:
        if 'name' in f and re.search(file_pattern, f['name'], re.IGNORECASE):
            print(f"     📂 Archivo encontrado en Drive: {f['name']}")
            return f
    print(f"     ⚠️ No se encontró ningún archivo que coincida con '{file_pattern}'.")
    return None

def subir_archivo_silencioso(service, local_path, folder_id):
    # (Función sin cambios)
    try:
        file_metadata = {'name': os.path.basename(local_path), 'parents': [folder_id]}
        media = MediaFileUpload(local_path, resumable=True)
        service.files().create(body=file_metadata, media_body=media, fields='id').execute()
        return True
    except Exception: return False

def normalizar_audio_silencioso(ffmpeg_path, input_path, output_path):
    # (Función sin cambios)
    try:
        cmd_pass1 = [ffmpeg_path, '-i', input_path, '-af', 'loudnorm=I=-23:LRA=7:tp=-2:print_format=json', '-f', 'null', '-']
        resultado_pass1 = subprocess.run(cmd_pass1, capture_output=True, text=True, encoding='utf-8')
        lines = resultado_pass1.stderr.strip().split('\n')
        json_str = "".join(lines[lines.index('{'):lines.index('}')+1])
        params = json.loads(json_str)
        filtro_loudnorm = f"loudnorm=I=-23:LRA=7:tp=-2:measured_I={params['input_i']}:measured_LRA={params['input_lra']}:measured_tp={params['input_tp']}:measured_thresh={params['input_thresh']}:offset={params['target_offset']}"
        cmd_pass2 = [ffmpeg_path, '-y', '-i', input_path, '-af', filtro_loudnorm, output_path]
        subprocess.run(cmd_pass2, capture_output=True)
        return True
    except Exception: return False

def visualizar_pistas_silencioso(pistas, segmentos, titulo_grafico, ruta_salida):
    """
    Función completa que lee los audios y genera la gráfica con las ondas y segmentos.
    """
    try:
        num_pistas = len(pistas)
        fig, axs = plt.subplots(num_pistas, 1, figsize=(25, 4 * num_pistas), sharex=True)
        fig.suptitle(titulo_grafico, fontsize=16, fontweight='bold')
        max_duration = 0

        # Asegurarse de que axs sea siempre un array para poder iterar
        if num_pistas == 1:
            axs = [axs]

        for i, (nombre_pista, ruta_pista) in enumerate(pistas.items()):
            ax = axs[i]
            try:
                data, rate = sf.read(ruta_pista)
                if data.ndim > 1: data = data[:, 0] # Convertir a mono

                duration = len(data) / rate
                max_duration = max(max_duration, duration)
                time = np.linspace(0., duration, num=len(data))

                # Optimización: No dibujar todos los puntos si el audio es largo
                step = max(1, len(data) // 5000)

                ax.plot(time[::step], data[::step], label=nombre_pista, color='cornflowerblue')
                ax.set_title(nombre_pista, fontsize=10)
                ax.grid(True, linestyle='--', alpha=0.6)
                ax.margins(x=0)
                ax.set_yticks([]) # Quitar los ejes Y (amplitud) para más limpieza

                # Dibujar los segmentos de audiodescripción (los "huecos")
                for idx, seg in enumerate(segmentos):
                    ax.axvspan(seg['start'], seg['end_real'], color='orange', alpha=0.3)
                    ax.text((seg['start'] + seg['end_real']) / 2, 0, f"AD {idx+1}",
                            ha='center', va='center', fontsize=9, bbox=dict(facecolor='white', alpha=0.5))

            except Exception as e:
                ax.text(0.5, 0.5, f"Error al cargar {os.path.basename(ruta_pista)}: {e}", ha='center', color='red')

        axs[-1].set_xlabel("Tiempo (segundos)")
        plt.xlim(0, max_duration)
        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        plt.savefig(ruta_salida, dpi=100)

    except Exception as e:
        print(f"❌ Error mayor al crear la gráfica: {e}")
    finally:
        plt.close(fig)

# --- 2. FUNCIÓN DE LIMPIEZA AÑADIDA ---
def limpiar_y_recortar_audio(ruta_entrada, ruta_salida):
    """
    Limpia el ruido de fondo y recorta los silencios al inicio y al final.
    """
    try:
        data, rate = sf.read(ruta_entrada)
        # Asegurarse de que el audio sea mono para la reducción de ruido
        if data.ndim > 1:
            data = data.mean(axis=1)

        ruido_reducido_data = nr.reduce_noise(y=data, sr=rate, prop_decrease=0.8)

        # Guardar temporalmente el archivo sin ruido
        ruta_temp_nr = os.path.join(LOCAL_TEMP_DIR, "temp_noise_reduced.wav")
        sf.write(ruta_temp_nr, ruido_reducido_data, rate)

        # Cargar con Pydub para recortar silencios
        audio = AudioSegment.from_wav(ruta_temp_nr)
        silence_threshold = audio.dBFS - 16

        trim_leading = detect_leading_silence(audio, silence_threshold=silence_threshold)
        audio_trimmed_start = audio[trim_leading:]

        trim_trailing = detect_leading_silence(audio_trimmed_start.reverse(), silence_threshold=silence_threshold)
        audio_trimmed_final = audio_trimmed_start.reverse()[trim_trailing:].reverse()

        # Aplicar un suave fade in/out
        audio_final = audio_trimmed_final.fade_in(30).fade_out(30)
        audio_final.export(ruta_salida, format="wav")

        if os.path.exists(ruta_temp_nr):
            os.remove(ruta_temp_nr)
        return True
    except Exception as e:
        tqdm.write(f"    ❌ Error durante la limpieza del audio: {e}")
        return False

# --- FUNCIÓN DE MEZCLA PRINCIPAL (LÓGICA RESTAURADA) ---
def mezclar_y_visualizar_todo(service, id_input, id_ad, id_output, id_evaluacion, locutores_a_mezclar):
    print("\n" + "="*80)
    print("###      🎧 INICIANDO PROCESO DE MEZCLA FINAL      ###")
    print("="*80)

    try:
        ffmpeg_path = imageio_ffmpeg.get_ffmpeg_exe()

        print("\n⚙️  FASE 1: PREPARACIÓN Y ANÁLISIS DE PISTAS BASE")
        print("  📥 Buscando y descargando archivos base...")
        video_file_info = find_file_in_drive(service, id_input, r'\.(mp4|mov)$')
        json_file_info = find_file_in_drive(service, id_input, r'Main_Video\.json$')
        if not video_file_info or not json_file_info: return

        local_video_path = descargar_archivo(service, video_file_info['id'], os.path.join(LOCAL_TEMP_DIR, video_file_info['name']))
        local_json_path = descargar_archivo(service, json_file_info['id'], os.path.join(LOCAL_TEMP_DIR, json_file_info['name']))
        audio_original_path = os.path.join(LOCAL_TEMP_DIR, "audio_original.wav")
        subprocess.run([ffmpeg_path, '-y', '-i', local_video_path, '-vn', '-acodec', 'pcm_s16le', '-ar', '44100', '-ac', '2', audio_original_path], capture_output=True)

        print("  🤖 Separando Audio con Demucs (esto puede tardar)...")
        subprocess.run(['demucs', '--two-stems=vocals', '-o', LOCAL_TEMP_DIR, audio_original_path], capture_output=True, check=True)

        print("  🔊 Normalizando pistas base (Narración y Fondo)...")
        ruta_no_vocals = glob.glob(os.path.join(LOCAL_TEMP_DIR, '**', 'no_vocals.wav'), recursive=True)[0]
        ruta_vocals_original = glob.glob(os.path.join(LOCAL_TEMP_DIR, '**', 'vocals.wav'), recursive=True)[0]
        ruta_no_vocals_norm = os.path.join(LOCAL_TEMP_DIR, "no_vocals_norm.wav")
        ruta_vocals_original_norm = os.path.join(LOCAL_TEMP_DIR, "vocals_original_norm.wav")
        normalizar_audio_silencioso(ffmpeg_path, ruta_no_vocals, ruta_no_vocals_norm)
        normalizar_audio_silencioso(ffmpeg_path, ruta_vocals_original, ruta_vocals_original_norm)

        # --- LÍNEA CORREGIDA ---
        # Esta línea se había borrado y ahora está restaurada
        with open(local_json_path, 'r', encoding='utf-8') as f:
            segmentos_json = sorted(json.load(f), key=lambda x: x.get('start', 0))
        # --- FIN DE LA CORRECCIÓN ---

        audios_ad_drive = [f for f in listar_archivos(service, id_ad) if f['name'].lower().endswith('.wav')]
        for audio_info in tqdm(audios_ad_drive, desc="  📥 Descargando audios AD"):
            descargar_archivo(service, audio_info['id'], os.path.join(LOCAL_TEMP_DIR, audio_info['name']))
        print("  ✅ Preparación completada.")

        print("\n🎤 FASE 2: PROCESAMIENTO POR LOCUTOR")
        for locutor in tqdm(locutores_a_mezclar, desc="  Progreso General"):
            tqdm.write("\n" + "-"*60 + f"\n  Procesando a: {locutor.upper()}\n" + "-"*60)

            dialogo_data = []
            segmentos_a_procesar = [s for s in segmentos_json if any(f"segmento_{s.get('id', '')}_{locutor}" in f_name for f_name in os.listdir(LOCAL_TEMP_DIR))]

            for s in tqdm(segmentos_a_procesar, desc=f"    🧼 Limpiando audios de {locutor}", leave=False):
                pattern = f"segmento_{s.get('id', '')}_"
                encontrado = next((f for f in os.listdir(LOCAL_TEMP_DIR) if pattern in f and locutor in f and f.lower().endswith('.wav')), None)

                if encontrado:
                    ruta_encontrada = os.path.join(LOCAL_TEMP_DIR, encontrado)
                    ruta_limpia = os.path.join(LOCAL_TEMP_DIR, f"clean_{encontrado}")

                    if limpiar_y_recortar_audio(ruta_encontrada, ruta_limpia):
                        duracion_real = AudioSegment.from_wav(ruta_limpia).duration_seconds
                        dialogo_data.append({
                            'path': ruta_limpia,
                            'start': s.get('start'),
                            'end_real': s.get('start') + duracion_real
                        })
                    else:
                        tqdm.write(f"      ⚠️ Falló la limpieza para el segmento {s.get('id', '')}. Saltando.")

            if not dialogo_data:
                tqdm.write(f"    ⚠️ No se encontraron audios válidos para {locutor}. Saltando...")
                continue

            tqdm.write("    🎚️  Creando 'huecos' en la narración original...")
            nar_original_con_huecos_path = os.path.join(LOCAL_TEMP_DIR, f"nar_huecos_{locutor}.wav")
            volume_intervals = "+".join([f"between(t,{d['start']},{d['end_real']})" for d in dialogo_data])
            filtro_volumen = f"volume=enable='{volume_intervals}':volume=0"
            subprocess.run([ffmpeg_path, '-y', '-i', ruta_vocals_original_norm, '-af', filtro_volumen, nar_original_con_huecos_path], capture_output=True)

            tqdm.write("    🎚️  Preparando pista de audiodescripción sincronizada...")
            ad_sin_norm_path = os.path.join(LOCAL_TEMP_DIR, f"ad_sin_norm_{locutor}.wav")
            nombre_salida_ad_timed = f"Audiodescripcion_{locutor}_TIMED_ONLY.wav"
            ruta_salida_ad_timed = os.path.join(LOCAL_OUTPUT_DIR, nombre_salida_ad_timed)
            cmd_ad = [ffmpeg_path, '-y', '-i', audio_original_path]
            filter_ad, streams_ad = ["[0:a]volume=0[base]"], "[base]"
            for i, d in enumerate(dialogo_data):
                cmd_ad.extend(['-i', d['path']])
                filter_ad.append(f"[{i+1}:a]adelay={int(d['start']*1000)}|{int(d['start']*1000)}[d{i}]")
                streams_ad += f"[d{i}]"
            filter_ad.append(f"{streams_ad}amix=inputs={len(dialogo_data)+1}[ad_track]")
            cmd_ad.extend(['-filter_complex', ";".join(filter_ad), '-map', '[ad_track]', ad_sin_norm_path])
            subprocess.run(cmd_ad, capture_output=True)
            normalizar_audio_silencioso(ffmpeg_path, ad_sin_norm_path, ruta_salida_ad_timed)
            subir_archivo_silencioso(service, ruta_salida_ad_timed, id_output)
            tqdm.write("      📤 Archivo 'TIMED_ONLY' subido.")

            tqdm.write("    🎚️  Mezclando todas las pistas finales...")
            nombre_salida_final = f"Audiodescripcion_{locutor}_MEZCLA_FINAL.wav"
            ruta_salida_final = os.path.join(LOCAL_OUTPUT_DIR, nombre_salida_final)

            # Filtro con ajuste de volumen para la música de fondo
            filtro_complejo_final = (
                f"[0:a]volume=0.25[bg_music];"
                f"[bg_music][1:a][2:a]amix=inputs=3,loudnorm[final]"
            )

            cmd_final = [
                ffmpeg_path, '-y',
                '-i', ruta_no_vocals_norm,         # [0:a] Fondo Musical
                '-i', nar_original_con_huecos_path, # [1:a] Narración con Huecos
                '-i', ruta_salida_ad_timed,       # [2:a] Nueva Audiodescripción
                '-filter_complex', filtro_complejo_final,
                '-map', '[final]', ruta_salida_final
            ]
            resultado_final = subprocess.run(cmd_final, capture_output=True, text=True, encoding='utf-8')

            if resultado_final.returncode == 0:
                tqdm.write("      ✅ Mezcla final generada.")
                subir_archivo_silencioso(service, ruta_salida_final, id_output)
                tqdm.write("      📤 Archivo 'MEZCLA_FINAL' subido.")

                tqdm.write("    📊 Generando visualización de pistas...")
                pistas_a_visualizar = {
                    "1. Narración Original (Completa)": ruta_vocals_original_norm,
                    "2. Narración con Huecos": nar_original_con_huecos_path,
                    "3. Fondo Musical": ruta_no_vocals_norm,
                    "4. Nueva Audiodescripción": ruta_salida_ad_timed,
                    "5. MEZCLA FINAL": ruta_salida_final
                }
                ruta_grafico = os.path.join(LOCAL_OUTPUT_DIR, f"Visualizacion_Mezcla_{locutor}.png")
                visualizar_pistas_silencioso(pistas_a_visualizar, dialogo_data, f"Análisis de Pistas - {locutor}", ruta_grafico)
                subir_archivo_silencioso(service, ruta_grafico, id_evaluacion)
                tqdm.write("      📤 Gráfico de visualización subido.")
            else:
                tqdm.write(f"    ❌ ERROR al generar la mezcla final para {locutor}.\n{resultado_final.stderr}")

    except Exception:
        import traceback; traceback.print_exc()
    finally:
        print("\n🧹 FASE 3: LIMPIEZA DE ARCHIVOS TEMPORALES")
        # (lógica de limpieza)
        print("  ✅ Limpieza completada.")

    print("\n" + "#"*80 + "\n###      ✨ TODOS LOS PROCESOS DE MEZCLA FINALIZADOS      ###\n" + "#"*80)

# --- EJECUCIÓN ---

mezclar_y_visualizar_todo(
    service=drive_service,
    id_input=FOLDER_ID_INPUT,
    id_ad=FOLDER_ID_AD,
    id_output=FOLDER_ID_OUTPUT_MAIN,
    id_evaluacion=FOLDER_ID_EVALUACION,
    locutores_a_mezclar=["Andrea", "Sergio", "Sebastian"]
)



# **Generacion de Token API de Google**

In [ ]:
# @title
# from google_auth_oauthlib.flow import InstalledAppFlow
# import os

# # 1. Define el permiso necesario
# SCOPES = ['https://www.googleapis.com/auth/drive']

# # 2. Revisa si el archivo de credenciales existe
# if not os.path.exists('credentials.json'):
#     print("❌ Error: No se encuentra el archivo 'credentials.json'.")
#     print("Asegúrate de haber subido el archivo correcto (para 'Aplicación de escritorio').")
# else:
#     # 3. Inicia el flujo de autenticación
#     # --- LÍNEA CORREGIDA ---
#     # Se añade explícitamente el 'redirect_uri' para el flujo de "copiar y pegar".
#     flow = InstalledAppFlow.from_client_secrets_file(
#         'credentials.json',
#         SCOPES,
#         redirect_uri='urn:ietf:wg:oauth:2.0:oob'
#     )

#     # 4. Genera la URL de autorización
#     auth_url, _ = flow.authorization_url(prompt='consent')

#     # 5. Muestra la URL al usuario
#     print('Por favor, ve a esta URL para autorizar el acceso:')
#     print(auth_url)
#     print('')

#     # 6. Pide al usuario que pegue el código que recibirá
#     code = input('Pega el código de autorización que recibiste aquí: ')

#     # 7. Intercambia el código por las credenciales
#     try:
#         flow.fetch_token(code=code)
#         creds = flow.credentials

#         # 8. Guarda las credenciales
#         with open('token.json', 'w') as token_file:
#             token_file.write(creds.to_json())

#         print("\n" + "="*50)
#         print("✅ ¡ÉXITO! Se ha creado el archivo 'token.json'.")
#         print(f"   Tu REFRESH TOKEN es: {creds.refresh_token}")
#         print("="*50)
#     except Exception as e:
#         print(f"\n❌ Ocurrió un error al intercambiar el código: {e}")